Core pipeline execution on files3_first40s.tdms, fit 30–31 s, manual cycle [0,700]. Cells 31–34 are archived step searches. Cell 39 needs an unavailable separate fixed-rod TDMS file; later interactive and unrelated explorations are not run.

In [1]:
from pathlib import Path
from functools import partial

import numpy as np
import matplotlib.pyplot as plt
from nptdms import TdmsFile
from scipy.optimize import minimize, differential_evolution

In [2]:

# ── T correction matrix ───────────────────────────────────────────────────────
def T_Icor_Matrix():  # order: 0, 90, 45, 135, with 45 reflected
    ret = np.zeros([4, 4])
    ret[0][0] = 0.152  # transmitted
    ret[0][1] = 0
    ret[1][0] = 0
    ret[1][1] = 0.148  # reflected
    alpha = 0.455
    beta  = 0.009  # This is quite critical
    t     = 0.757  # transmitted in PBS
    r     = 0.741  # reflected
    ret[2][0] = -alpha * beta * r  # Reflected
    ret[2][1] =  alpha * beta * r
    ret[2][2] =  alpha * alpha * r
    ret[2][3] =  beta  * beta  * r
    ret[3][0] = -alpha * beta * t  # Transmitted
    ret[3][1] =  alpha * beta * t
    ret[3][2] =  beta  * beta  * t
    ret[3][3] =  alpha * alpha * t
    return np.linalg.inv(ret) / 8

# ── Gain fit ──────────────────────────────────────────────────────────────────
def _chcor_objective(params, ac, mat):
    a90, a45, a135 = params
    ap = np.copy(ac)
    ap[1, :] *= a90
    ap[2, :] *= a45
    ap[3, :] *= a135
    bc = np.dot(mat, ap)
    return np.sum((bc[0] + bc[1] - bc[2] - bc[3]) ** 2)

def find_best_coeff_using_mat(c0, c90, c45, c135, mat):
    ac = np.vstack((c0, c90, c45, c135)).astype(float)
    return minimize(partial(_chcor_objective, ac=ac, mat=mat), [1.0, 1.0, 1.0])

# ── Polarisation index mapping ────────────────────────────────────────────────
#   A small helper function to switch between different orders of polarizations 
#   depending on the conventions used.
_orientations = ["90", "45", "135", "0"]

def get_pol_ind(listpol):
    return [_orientations.index(p) for p in listpol]

# ── Pre-compute T matrix in stack order [c90, c45, c135, c0] ──────────────────
matcorb = T_Icor_Matrix()
_inds   = get_pol_ind(["0", "90", "45", "135"])
_Pmat   = np.zeros([4, 4])
for _i in range(4):
    _Pmat[_i, _inds[_i]] = 1
matcor = np.dot(np.linalg.inv(_Pmat), np.dot(matcorb, _Pmat))
del _inds, _Pmat, _i

# ── Anisotropy ────────────────────────────────────────────────────────────────
def anisotropy_from_channels(c0, c90, c45, c135):
    denom_x = c0 + c90
    denom_y = c45 + c135
    ax = np.where(denom_x > 0, (c0 - c90)   / denom_x, 0.0)
    ay = np.where(denom_y > 0, (c45 - c135) / denom_y, 0.0)
    return ax, ay

# ── Fourkas ───────────────────────────────────────────────────────────────────
def fourkas_ABC(alpha):
    """Coefficients A, B, C as functions of collection half-angle α."""
    ca = np.cos(alpha)
    A  = 1/6  - ca/4               + (ca**3)/12
    B  =        ca/8               - (ca**3)/8
    C  = 7/48 - ca/16 - (ca**2)/16 - (ca**3)/48
    return A, B, C

# ── Forward model: Rodrigues rotation + Fourkas ──────────────────────────────
def fourkas_template(Theta_axis_deg, Phi_axis_deg, Lambda_deg, A, B, C,
                     N_model=720):
    """Generate model anisotropy curve for given rotation geometry.

    Parameters
    ----------
    Theta_axis_deg, Phi_axis_deg : rotation axis direction (degrees)
    Lambda_deg                   : cone half-angle (degrees)
    A, B, C                      : Fourkas coefficients
    N_model                      : number of points on the model curve

    Returns
    -------
    ax_model, ay_model : arrays of shape (N_model,)
    """
    theta_ax = np.radians(Theta_axis_deg)
    phi_ax   = np.radians(Phi_axis_deg)
    lam      = np.radians(Lambda_deg)

    # Rotation axis unit vector
    kx = np.sin(theta_ax) * np.cos(phi_ax)
    ky = np.sin(theta_ax) * np.sin(phi_ax)
    kz = np.cos(theta_ax)
    k_hat = np.array([kx, ky, kz])

    # Orthonormal basis (k_hat, e1, e2)
    if abs(kz) < 0.9999:
        e1 = np.array([0.0, 0.0, 1.0]) - kz * k_hat
        e1 /= np.linalg.norm(e1)
    else:
        e1 = np.array([1.0, 0.0, 0.0])
    e2 = np.cross(k_hat, e1)

    # Rodrigues rotation: d(ψ) = cos(Λ)·k̂ + sin(Λ)·[cos(ψ)·e1 + sin(ψ)·e2]
    psi = np.linspace(0.0, 2.0 * np.pi, N_model, endpoint=False)
    d = (  np.cos(lam) * k_hat[np.newaxis, :]
         + np.sin(lam) * (np.cos(psi)[:, np.newaxis] * e1[np.newaxis, :]
                        + np.sin(psi)[:, np.newaxis] * e2[np.newaxis, :]) )

    theta_psi = np.arccos(np.clip(d[:, 2], -1.0, 1.0))
    phi_psi   = np.arctan2(d[:, 1], d[:, 0])

    # Forward Fourkas: θ → r → (ax, ay)
    s2 = np.sin(theta_psi) ** 2
    r  = C * s2 / (A + B * s2)
    ax_model = r * np.cos(2.0 * phi_psi)
    ay_model = r * np.sin(2.0 * phi_psi)

    return ax_model, ay_model

In [3]:
# ── File ──────────────────────────────────────────────────────────────────────
tdms_path = Path("files3_first40s.tdms")  # repository test sample; change for local data

# ── Output folder detection ──────────────────────────────────────────────────
# Detect "data" folder in the path and replace with "results"
_tdms_parts = tdms_path.parts
if "data" in _tdms_parts:
    _data_idx = _tdms_parts.index("data")
    _relative_path = Path(*_tdms_parts[_data_idx+1:])
    _output_dir = Path(*_tdms_parts[:_data_idx]) / "results" / _relative_path.with_suffix("")
else:
    _relative_path = Path(tdms_path.name)
    _output_dir = Path("results") / tdms_path.stem
_output_dir.mkdir(parents=True, exist_ok=True)
_relative_filename = str(_relative_path)

print(f"Output folder: {_output_dir}")
print(f"Relative name: {_relative_filename}")

# ── Channel-correction coefficients ──────────────────────────────────────────
# o, a all use the same physical stack order: [c90, c45, c135, c0]
#
# a: per-channel gain — set to None to auto-calculate from data
# a_manual = [1.08, 1.16, 1.02, 1.0]   # e.g. default [a_90, a_45, a_135, a_0] = [1.0, 1.2, 1.0, 1.0]
a_manual = None

# Time window for the coefficient fit (None = use all data)
fit_start_s    = 30.0
fit_end_s      = 31.0
dec_correction = 1   # decimation applied during the fit

# ── T-matrix correction toggle ───────────────────────────────────────────────
# Set to True to apply the optical T-matrix correction, False to skip it
use_tmatrix = True

print(f"File : {tdms_path.name}")
print(f"Exists: {tdms_path.exists()}")
print(f"T-matrix correction: {'ON' if use_tmatrix else 'OFF'}")

Output folder: results/files3_first40s
Relative name: files3_first40s.tdms
File : files3_first40s.tdms
Exists: True
T-matrix correction: ON


In [4]:

# ── Load raw channel intensities from TDMS file ───────────────────────────────
tdms_obj  = TdmsFile.open(str(tdms_path))
group     = tdms_obj.groups()[0]
channels  = group.channels()
datasize  = len(channels[0])

try:
    time_inc = float(channels[0].properties["wf_increment"])
    time_off = float(channels[0].properties["wf_start_offset"])
except KeyError:
    time_inc = 4e-6
    time_off = 0.0
    print("Warning: no time metadata — using time_inc=4e-6 s")

freq   = 1.0 / time_inc
time_s = time_off + time_inc * np.arange(datasize)

idx      = get_pol_ind(["0", "90", "45", "135"])
c0_raw   = np.asarray(channels[idx[0]][:], dtype=float)
c90_raw  = np.asarray(channels[idx[1]][:], dtype=float)
c45_raw  = np.asarray(channels[idx[2]][:], dtype=float)
c135_raw = np.asarray(channels[idx[3]][:], dtype=float)

print(f"Loaded: {tdms_path.name}")
print(f"  {datasize:,} points  |  {freq:.0f} Hz  |  duration: {time_s[-1]:.1f} s")


Loaded: files3_first40s.tdms
  10,000,000 points  |  250000 Hz  |  duration: 40.0 s


In [5]:
# ── Data preparation ──────────────────────────────────────────────────────────
# Decimate raw channels over the fit window and compute raw anisotropy.
N_SUBSAMPLE  = 90    # phase points fed to shape fitting (used by Section 8 too)
FORCE_UNIQUE = False # phase-lattice uniqueness (Section 8)

fit_s = int((fit_start_s - time_off) / time_inc) if fit_start_s is not None else 0
fit_e = int((fit_end_s   - time_off) / time_inc) if fit_end_s   is not None else datasize
fit_s = max(0, min(fit_s, datasize))
fit_e = max(fit_s, min(fit_e, datasize))

_c0_dec   = c0_raw [fit_s:fit_e:dec_correction]
_c90_dec  = c90_raw[fit_s:fit_e:dec_correction]
_c45_dec  = c45_raw[fit_s:fit_e:dec_correction]
_c135_dec = c135_raw[fit_s:fit_e:dec_correction]
_t_dec    = time_s [fit_s:fit_e:dec_correction]

_ax_raw, _ay_raw = anisotropy_from_channels(_c0_dec, _c90_dec, _c45_dec, _c135_dec)
if len(_t_dec) > 0:
    print(f"Fit window: {len(_ax_raw)} decimated samples, "
          f"t = {_t_dec[0]:.3f} – {_t_dec[-1]:.3f} s")

from scipy.spatial import cKDTree as _OB_KDTree
from scipy.signal import find_peaks as _OB_find_peaks, peak_widths as _OB_peak_widths
from scipy.signal import savgol_filter as _OB_savgol
from scipy.interpolate import splprep as _OB_splprep, splev as _OB_splev

# --- SCMS knobs (diagnostic background only) --------------------------------
SCMS_BANDWIDTH    = 0.020
SCMS_N_SEEDS      = 3000
SCMS_N_ITERS      = 25
SCMS_TOL          = 1e-5
SCMS_DATA_CAP     = 8000
SCMS_SEED_BATCH   = 500
SCMS_DENSITY_PCTL = 0
SCMS_RATIO_MIN    = 0
# --- connectivity filter: keep large connected components on ridge ----------
SCMS_CONNECT_RADIUS    = 0.01  # ridge pts within this distance are connected
SCMS_CONNECT_MIN_SIZE  = 6     # drop components with fewer than this many points (2 = drop singletons)
# --- cycle-detection knobs (ported from Fourkas_processing.ipynb) -----------
CYC_WINDOW        = 200
CYC_STEP          = 5
CYC_CLOSURE_FAC   = 1.0
CYC_SEARCH_LIMIT  = None
# --- cycle smoothing (Savgol on raw cycle, in time order, periodic) ---------
CYC_SAVGOL_WINDOW = 201   # odd, in raw samples (matches Fourkas_processing.ipynb)
CYC_SAVGOL_ORDER  = 3
# --- periodic B-spline smoothing of the closed cycle ------------------------
CYC_SPLINE_S      = 0.5   # splprep smoothing parameter (matches original)
# --- snap final spline curve onto SCMS ridge --------------------------------
CYC_SNAP_TO_SCMS    = True
# --- post-snap smoothing of the merged track --------------------------------
CYC_POST_SMOOTH     = True
CYC_POST_SAVGOL_WIN = 51   # odd, in merged-track samples
CYC_POST_SAVGOL_ORDER = 3
CYC_POST_SPLINE_S   = 0.05  # final periodic B-spline smoothing on snapped track
# --- merge-track resolution -------------------------------------------------
MERGE_TRACK_N     = max(4 * N_SUBSAMPLE, 480)


def _ob_scms(pts, h, n_seeds, n_iters, tol, data_cap, seed_batch):
    rng = np.random.default_rng(0)
    n = len(pts)
    if n > data_cap:
        sub = rng.choice(n, size=data_cap, replace=False)
        data = pts[sub]
    else:
        data = pts
    seed_idx = rng.choice(len(data), size=min(n_seeds, len(data)), replace=False)
    seeds = data[seed_idx].copy()
    inv2h2 = 1.0 / (2.0 * h * h)
    final_ratio = np.zeros(len(seeds))

    def _step(s_batch, store_ratio, ratio_out, idx0):
        diff = data[None, :, :] - s_batch[:, None, :]
        r2 = np.sum(diff * diff, axis=2)
        w = np.exp(-r2 * inv2h2)
        w_sum = w.sum(axis=1, keepdims=True) + 1e-30
        m = (w[:, :, None] * data[None, :, :]).sum(axis=1) / w_sum
        wt = w[:, :, None]
        cov = np.einsum("nij,nik->njk", wt * diff, diff) / w_sum[:, :, None]
        evals, evecs = np.linalg.eigh(cov)
        v_minor = evecs[:, :, 0]
        proj = np.sum((m - s_batch) * v_minor, axis=1, keepdims=True) * v_minor
        if store_ratio:
            ratio_out[idx0:idx0 + len(s_batch)] = (
                evals[:, 1] / np.maximum(evals[:, 0], 1e-30)
            )
        return s_batch + proj

    for it in range(n_iters):
        seeds_new = np.empty_like(seeds)
        store = (it == n_iters - 1)
        for b0 in range(0, len(seeds), seed_batch):
            b1 = min(b0 + seed_batch, len(seeds))
            seeds_new[b0:b1] = _step(seeds[b0:b1], store, final_ratio, b0)
        max_step = np.max(np.linalg.norm(seeds_new - seeds, axis=1))
        seeds = seeds_new
        if max_step < tol:
            if not store:
                for b0 in range(0, len(seeds), seed_batch):
                    b1 = min(b0 + seed_batch, len(seeds))
                    _step(seeds[b0:b1], True, final_ratio, b0)
            break

    density = np.empty(len(seeds))
    for b0 in range(0, len(seeds), seed_batch):
        b1 = min(b0 + seed_batch, len(seeds))
        diff = data[None, :, :] - seeds[b0:b1, None, :]
        r2 = np.sum(diff * diff, axis=2)
        density[b0:b1] = np.exp(-r2 * inv2h2).sum(axis=1)
    return seeds, final_ratio, density


def _ob_detect_cycle(ax, ay, window=200, search_limit=None, step=5, closure_factor=1.0):
    traj = np.column_stack([ax, ay])
    n = len(traj)
    if search_limit is None:
        search_limit = min(n // 2, 100_000)
    start_idx = 0
    ref_tree = _OB_KDTree(traj[start_idx:start_idx + window])
    n_sample = max(2, window // 10)
    dists_close = np.linalg.norm(
        traj[start_idx] - traj[start_idx:start_idx + n_sample], axis=1)
    dist_thresh = np.median(dists_close) * closure_factor
    i_list = np.arange(start_idx + window, start_idx + search_limit, step)
    if len(i_list) == 0:
        return start_idx, min(start_idx + search_limit, n)
    scores = []
    for i in i_list:
        end = min(i + window, n)
        tree2 = _OB_KDTree(traj[i:end])
        scores.append(tree2.count_neighbors(ref_tree, dist_thresh))
    scores = np.array(scores, dtype=float)
    prom_div = 2
    peaks = np.array([], dtype=int)
    while len(peaks) == 0 and prom_div < 128:
        peaks = _OB_find_peaks(scores, prominence=window / prom_div)[0]
        prom_div *= 2
    if len(peaks) == 0:
        return start_idx, int(i_list[np.argmax(scores)])
    _, _, _, right_ips = _OB_peak_widths(scores, peaks, rel_height=0.5)
    right_ips = np.round(right_ips).astype(int)
    peak_ind = 0
    while peak_ind < len(peaks) - 1:
        if np.min(scores[: peaks[peak_ind]]) > 2 * np.min(scores):
            peak_ind += 1
        else:
            break
    for k in range(peak_ind, min(peak_ind + 3, len(peaks))):
        if scores[peaks[k]] > 1.5 * scores[peaks[peak_ind]]:
            peak_ind = k
    end_idx = i_list[min(right_ips[peak_ind], len(i_list) - 1)]
    return int(start_idx), int(end_idx)


def _ob_smooth_cycle_2d(ax_seg, ay_seg, smoothing=0.5, num_points=500,
                        resample_n=500):
    """Periodic B-spline smooth of a 2-D cycle (matches Fourkas_processing.ipynb)."""
    pts = np.column_stack([ax_seg, ay_seg])
    if not np.allclose(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])
    diffs = np.diff(pts, axis=0)
    seg_len = np.hypot(diffs[:, 0], diffs[:, 1])
    cum_arc = np.concatenate([[0], np.cumsum(seg_len)])
    total_arc = cum_arc[-1]
    if total_arc == 0:
        return ax_seg, ay_seg
    arc_uniform = np.linspace(0, total_arc, resample_n, endpoint=False)
    rx = np.interp(arc_uniform, cum_arc, pts[:, 0], period=total_arc)
    ry = np.interp(arc_uniform, cum_arc, pts[:, 1], period=total_arc)
    pts_uniform = np.column_stack([rx, ry])
    pts_uniform = np.vstack([pts_uniform, pts_uniform[0]])
    if smoothing is None:
        smoothing = 0.01 * resample_n
    tck, _ = _OB_splprep(pts_uniform.T, per=True, s=smoothing)
    u = np.linspace(0, 1, num_points, endpoint=False)
    sx, sy = _OB_splev(u, tck)
    return np.asarray(sx), np.asarray(sy)


# 1) SCMS ridge (diagnostic shape, background only)
_ob_pool = np.column_stack([_ax_raw, _ay_raw])
_ob_pool = _ob_pool[np.all(np.isfinite(_ob_pool), axis=1)]
_ob_seeds, _ob_ratio, _ob_density = _ob_scms(
    _ob_pool, SCMS_BANDWIDTH, SCMS_N_SEEDS, SCMS_N_ITERS, SCMS_TOL,
    SCMS_DATA_CAP, SCMS_SEED_BATCH,
)
_ob_d_thr = np.percentile(_ob_density, SCMS_DENSITY_PCTL)
_ob_keep = (_ob_density >= _ob_d_thr) & (_ob_ratio >= SCMS_RATIO_MIN)
_ob_ridge_pre = _ob_seeds[_ob_keep]
print(f"[Option B] SCMS: kept {_ob_keep.sum()}/{len(_ob_seeds)} ridge pts  "
      f"(h={SCMS_BANDWIDTH}, iters={SCMS_N_ITERS})")

# --- connectivity filter: keep large connected components -------------------
from scipy.sparse import csr_matrix as _OB_csr
from scipy.sparse.csgraph import connected_components as _OB_cc
if len(_ob_ridge_pre) > 1:
    _ob_ctree = _OB_KDTree(_ob_ridge_pre)
    _ob_pairs = _ob_ctree.query_pairs(r=SCMS_CONNECT_RADIUS, output_type="ndarray")
    _ob_nr = len(_ob_ridge_pre)
    if len(_ob_pairs):
        rows = np.concatenate([_ob_pairs[:, 0], _ob_pairs[:, 1]])
        cols = np.concatenate([_ob_pairs[:, 1], _ob_pairs[:, 0]])
        data = np.ones(len(rows), dtype=np.uint8)
        _ob_graph = _OB_csr((data, (rows, cols)), shape=(_ob_nr, _ob_nr))
    else:
        _ob_graph = _OB_csr((_ob_nr, _ob_nr))
    _ob_ncomp, _ob_labels = _OB_cc(_ob_graph, directed=False)
    _ob_sizes = np.bincount(_ob_labels)
    _ob_keep_lbl = np.where(_ob_sizes >= SCMS_CONNECT_MIN_SIZE)[0]
    _ob_cc_mask = np.isin(_ob_labels, _ob_keep_lbl)
    _ob_ridge = _ob_ridge_pre[_ob_cc_mask]
    print(f"[Option B] connectivity: {_ob_ncomp} comps, top sizes="
          f"{sorted(_ob_sizes.tolist(), reverse=True)[:5]}, kept "
          f"{_ob_cc_mask.sum()}/{len(_ob_ridge_pre)} "
          f"(radius={SCMS_CONNECT_RADIUS}, min_size={SCMS_CONNECT_MIN_SIZE})")
else:
    _ob_ridge = _ob_ridge_pre
    _ob_cc_mask = np.ones(len(_ob_ridge_pre), dtype=bool)

# -- Diagnostics: SCMS ridge --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
ax.scatter(_ax_raw, _ay_raw, s=1, alpha=0.06, c='grey', rasterized=True, label='raw')
ax.scatter(_ob_seeds[~_ob_keep, 0], _ob_seeds[~_ob_keep, 1], s=4, c='red', alpha=0.4,
           label='dropped (density/ratio)')
ax.scatter(_ob_ridge_pre[~_ob_cc_mask, 0], _ob_ridge_pre[~_ob_cc_mask, 1], s=6, c='orange',
           alpha=0.7, label='dropped (connectivity)')
ax.scatter(_ob_ridge[:, 0], _ob_ridge[:, 1], s=5, c='tab:blue', alpha=0.7,
           label='SCMS ridge (kept)')
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect('equal')
ax.set_xlabel('ax'); ax.set_ylabel('ay'); ax.grid(alpha=0.25)
ax.set_title(f'SCMS ridge (h={SCMS_BANDWIDTH}, kept {len(_ob_ridge)} pts)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


Fit window: 250000 decimated samples, t = 30.000 – 31.000 s
[Option B] SCMS: kept 3000/3000 ridge pts  (h=0.02, iters=25)
[Option B] connectivity: 708 comps, top sizes=[490, 137, 73, 72, 53], kept 1945/3000 (radius=0.01, min_size=6)


In [6]:
# 2) Cycle detection on RAW time-ordered trajectory
_ob_cyc_start, _ob_cyc_end = _ob_detect_cycle(
    _ax_raw, _ay_raw,
    window=CYC_WINDOW, step=CYC_STEP,
    search_limit=CYC_SEARCH_LIMIT, closure_factor=CYC_CLOSURE_FAC,
)
_ob_n_cyc = _ob_cyc_end - _ob_cyc_start
print(f"[Option B] cycle window: [{_ob_cyc_start}, {_ob_cyc_end}]  "
      f"({_ob_n_cyc} samples / {len(_ax_raw)})")
if _ob_n_cyc < 20:
    raise RuntimeError("Option B cycle window too short; check fit_start/end.")

# -- Manual cycle override (optional fallback) -------------------------------
# Set CYC_MANUAL_OVERRIDE = True to enable manual cycle selection. Then:
#   - If CYC_MANUAL_START and CYC_MANUAL_END are BOTH set (not None), use them
#     directly — no slider. Use this on re-runs once you know the indices.
#   - If either is None, display interactive sliders to pick the indices. After
#     clicking 'Apply selection', copy the printed start/end into
#     CYC_MANUAL_START / CYC_MANUAL_END so the next run skips the slider.
CYC_MANUAL_OVERRIDE = True
CYC_MANUAL_START = 0   # int index, or None to use the slider
CYC_MANUAL_END   = 700   # int index, or None to use the slider

if CYC_MANUAL_OVERRIDE and (CYC_MANUAL_START is not None) and (CYC_MANUAL_END is not None):
    # Direct manual indices — no slider.
    _ob_cyc_start = int(CYC_MANUAL_START)
    _ob_cyc_end   = int(max(_ob_cyc_start + 1, int(CYC_MANUAL_END)))
    _ob_cyc_start = int(np.clip(_ob_cyc_start, 0, len(_ax_raw) - 2))
    _ob_cyc_end   = int(np.clip(_ob_cyc_end, _ob_cyc_start + 1, len(_ax_raw)))
    _ob_n_cyc = _ob_cyc_end - _ob_cyc_start
    if _ob_n_cyc < 20:
        raise RuntimeError(
            f"Manual cycle window too short ({_ob_n_cyc} samples). "
            "Widen CYC_MANUAL_START / CYC_MANUAL_END."
        )
    print(f"Manual cycle indices: [{_ob_cyc_start}, {_ob_cyc_end}] "
          f"({_ob_n_cyc} samples). Slider skipped.")
elif CYC_MANUAL_OVERRIDE:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    print("Manual override ON but no indices set — use the slider, then paste the "
          "printed values into CYC_MANUAL_START / CYC_MANUAL_END for next time.")

    _n_dec = len(_ax_raw)
    _slider_start = widgets.IntSlider(
        value=_ob_cyc_start, min=0, max=_n_dec - 1, step=1,
        description='Start idx:', continuous_update=False,
        layout=widgets.Layout(width='90%'),
        style={'description_width': '80px'},
    )
    _slider_end = widgets.IntSlider(
        value=_ob_cyc_end, min=1, max=_n_dec, step=1,
        description='End idx:', continuous_update=False,
        layout=widgets.Layout(width='90%'),
        style={'description_width': '80px'},
    )
    _plot_out = widgets.Output()
    _status_out = widgets.Output()

    def _redraw(*_args):
        s, e = _slider_start.value, max(_slider_start.value + 1, _slider_end.value)
        with _plot_out:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
            axes[0].scatter(_ax_raw, _ay_raw, s=1, alpha=0.1, c='grey',
                            rasterized=True, label='all decimated')
            axes[0].scatter(_ax_raw[s:e], _ay_raw[s:e], s=2, alpha=0.5,
                            c='tab:blue', rasterized=True, label='selected')
            axes[0].set(xlabel='ax', ylabel='ay', title='Cycle selection',
                        xlim=(-1, 1), ylim=(-1, 1), aspect='equal')
            axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

            axes[1].plot(_ax_raw[s:e], '-', lw=0.6, c='tab:blue', label='ax')
            axes[1].plot(_ay_raw[s:e], '-', lw=0.6, c='tab:orange', label='ay')
            axes[1].set(xlabel='sample (within selection)', title='Time-ordered ax/ay')
            axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)
            plt.tight_layout(); plt.show()

    def _apply(btn):
        global _ob_cyc_start, _ob_cyc_end, _ob_n_cyc
        _ob_cyc_start = int(_slider_start.value)
        _ob_cyc_end   = int(max(_ob_cyc_start + 1, _slider_end.value))
        _ob_n_cyc = _ob_cyc_end - _ob_cyc_start
        with _status_out:
            clear_output(wait=True)
            print(f"Applied manual cycle: [{_ob_cyc_start}, {_ob_cyc_end}]  "
                  f"({_ob_n_cyc} samples).  Re-run Section 9 to rebuild the merged track.")
            print(f"To skip the slider next time, set:")
            print(f"    CYC_MANUAL_START = {_ob_cyc_start}")
            print(f"    CYC_MANUAL_END   = {_ob_cyc_end}")

    # Clear observers left over from a previous run of this cell before
    # re-registering — otherwise re-running stacks duplicate listeners and
    # each slider move fires several redraws (multiple plots appear).
    _slider_start.unobserve_all()
    _slider_end.unobserve_all()
    _slider_start.observe(_redraw, names='value')
    _slider_end.observe(_redraw, names='value')

    _apply_btn = widgets.Button(description='Apply selection', button_style='success',
                                icon='check', layout=widgets.Layout(width='200px'))
    _apply_btn.on_click(_apply)

    _redraw()
    display(widgets.VBox([_slider_start, _slider_end, _apply_btn, _status_out, _plot_out]))
else:
    print(f"Manual override disabled — using auto-detected cycle "
          f"[{_ob_cyc_start}, {_ob_cyc_end}] ({_ob_n_cyc} samples).")

# -- Diagnostics: cycle window on raw cloud ---------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
axes[0].scatter(_ax_raw, _ay_raw, s=1, alpha=0.06, c='grey', rasterized=True, label='all raw')
axes[0].scatter(_ax_raw[_ob_cyc_start:_ob_cyc_end], _ay_raw[_ob_cyc_start:_ob_cyc_end],
                s=2, c=np.linspace(0, 1, _ob_n_cyc), cmap='hsv', alpha=0.7, label='cycle')
axes[0].set_xlim(-1, 1); axes[0].set_ylim(-1, 1); axes[0].set_aspect('equal')
axes[0].set_xlabel('ax'); axes[0].set_ylabel('ay'); axes[0].grid(alpha=0.25)
axes[0].set_title(f'Cycle [{_ob_cyc_start}, {_ob_cyc_end}] ({_ob_n_cyc} samples)')
axes[0].legend(fontsize=8)
axes[1].plot(_ax_raw[_ob_cyc_start:_ob_cyc_end], '-', lw=0.6, c='tab:blue', label='ax')
axes[1].plot(_ay_raw[_ob_cyc_start:_ob_cyc_end], '-', lw=0.6, c='tab:orange', label='ay')
axes[1].set_xlabel('sample (within cycle)'); axes[1].grid(alpha=0.25)
axes[1].set_title('Time-ordered ax/ay over cycle')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

[Option B] cycle window: [0, 9380]  (9380 samples / 250000)
Manual cycle indices: [0, 700] (700 samples). Slider skipped.


In [7]:
# 3) Periodic Savgol smoothing of the raw cycle slice (in time order)
_ob_cx_raw = _ax_raw[_ob_cyc_start:_ob_cyc_end]
_ob_cy_raw = _ay_raw[_ob_cyc_start:_ob_cyc_end]
_ob_sg_win = min(CYC_SAVGOL_WINDOW, _ob_n_cyc // 2 * 2 - 1)
if _ob_sg_win < CYC_SAVGOL_ORDER + 2:
    _ob_sg_win = CYC_SAVGOL_ORDER + 2
    if _ob_sg_win % 2 == 0:
        _ob_sg_win += 1
_ob_cx_sm = _OB_savgol(_ob_cx_raw, _ob_sg_win, CYC_SAVGOL_ORDER, mode="wrap")
_ob_cy_sm = _OB_savgol(_ob_cy_raw, _ob_sg_win, CYC_SAVGOL_ORDER, mode="wrap")
print(f"[Option B] Savgol smoothing: window={_ob_sg_win} samples, "
      f"order={CYC_SAVGOL_ORDER}")

# 4) Periodic B-spline smoothing of the closed cycle (matches original)
_ob_sx_spl, _ob_sy_spl = _ob_smooth_cycle_2d(
    _ob_cx_sm, _ob_cy_sm,
    smoothing=CYC_SPLINE_S,
    num_points=MERGE_TRACK_N,
    resample_n=max(MERGE_TRACK_N, 500),
)

# 5) Snap smooth spline onto SCMS ridge (geometry from SCMS, ordering from cycle)
if CYC_SNAP_TO_SCMS and len(_ob_ridge) > 0:
    _ob_tree = _OB_KDTree(_ob_ridge)
    _, _ob_nn = _ob_tree.query(np.column_stack([_ob_sx_spl, _ob_sy_spl]))
    _sx_merge_track = _ob_ridge[_ob_nn, 0]
    _sy_merge_track = _ob_ridge[_ob_nn, 1]
    print(f"[Option B] merged track: {len(_sx_merge_track)} pts (spline -> snapped to SCMS ridge)")
else:
    _sx_merge_track, _sy_merge_track = _ob_sx_spl, _ob_sy_spl
    print(f"[Option B] merged track: {len(_sx_merge_track)} pts (periodic B-spline, s={CYC_SPLINE_S})")
_touch_pt = None  # not used by Option B

# 6) Post-snap smoothing: Savgol (periodic) + final B-spline on the merged track
if CYC_POST_SMOOTH and len(_sx_merge_track) > CYC_POST_SAVGOL_ORDER + 2:
    _ob_post_n = len(_sx_merge_track)
    _ob_post_win = min(CYC_POST_SAVGOL_WIN, _ob_post_n - (1 - _ob_post_n % 2))
    if _ob_post_win % 2 == 0:
        _ob_post_win -= 1
    if _ob_post_win < CYC_POST_SAVGOL_ORDER + 2:
        _ob_post_win = CYC_POST_SAVGOL_ORDER + 2
        if _ob_post_win % 2 == 0:
            _ob_post_win += 1
    _sx_merge_track = _OB_savgol(_sx_merge_track, _ob_post_win, CYC_POST_SAVGOL_ORDER, mode="wrap")
    _sy_merge_track = _OB_savgol(_sy_merge_track, _ob_post_win, CYC_POST_SAVGOL_ORDER, mode="wrap")
    _sx_merge_track, _sy_merge_track = _ob_smooth_cycle_2d(
        _sx_merge_track, _sy_merge_track,
        smoothing=CYC_POST_SPLINE_S,
        num_points=MERGE_TRACK_N,
        resample_n=max(MERGE_TRACK_N, 500),
    )
    print(f"[Option B] post-smooth: Savgol win={_ob_post_win}, spline s={CYC_POST_SPLINE_S} -> {len(_sx_merge_track)} pts")

# -- Diagnostics: merged track ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
_ob_t_color = np.linspace(0, 1, _ob_n_cyc)
axes[0].scatter(_ax_raw, _ay_raw, s=1, alpha=0.05, c='grey', rasterized=True)
axes[0].scatter(_ob_cx_raw, _ob_cy_raw, s=2, c='black', alpha=0.15, label='raw cycle')
axes[0].scatter(_ob_cx_sm, _ob_cy_sm, s=4, c=_ob_t_color, cmap='hsv', alpha=0.85,
                label=f'Savgol (win={_ob_sg_win})')
axes[0].set_xlim(-1, 1); axes[0].set_ylim(-1, 1); axes[0].set_aspect('equal')
axes[0].set_xlabel('ax'); axes[0].set_ylabel('ay'); axes[0].grid(alpha=0.25)
axes[0].set_title('Cycle slice + Savgol smoothing'); axes[0].legend(fontsize=8)

axes[1].scatter(_ax_raw, _ay_raw, s=1, alpha=0.06, c='grey', rasterized=True, label='raw')
axes[1].plot(_ob_sx_spl, _ob_sy_spl, '-', c='tab:orange', lw=1.0, alpha=0.6, label='spline (pre-snap)')
axes[1].plot(_sx_merge_track, _sy_merge_track, '-', c='tab:red', lw=1.4, alpha=0.95,
             label='merged track')
_ob_lat = np.linspace(0, len(_sx_merge_track), N_SUBSAMPLE, endpoint=False).astype(int)
axes[1].scatter(_sx_merge_track[_ob_lat], _sy_merge_track[_ob_lat],
                s=14, c=np.linspace(0, 1, N_SUBSAMPLE), cmap='hsv', alpha=0.95,
                label='phase preview')
axes[1].set_xlim(-1, 1); axes[1].set_ylim(-1, 1); axes[1].set_aspect('equal')
axes[1].set_xlabel('ax'); axes[1].set_ylabel('ay'); axes[1].grid(alpha=0.25)
axes[1].set_title(f'Merged track ({len(_sx_merge_track)} pts)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


[Option B] Savgol smoothing: window=201 samples, order=3
[Option B] merged track: 480 pts (spline -> snapped to SCMS ridge)
[Option B] post-smooth: Savgol win=51, spline s=0.05 -> 480 pts


In [8]:
# ── Phase-ordered matching from merged track (for shape fitting) ───────────────
# Build N_SUBSAMPLE evenly spaced phase points on the merged track, set phase 0 to
# the leftmost merged-track point, and match each phase point to its nearest raw
# point over the full fit window. The selected fit arrays are ordered by merged-track
# phase (not by original time/index order).

if ('_sx_merge_track' not in globals()) or (_sx_merge_track is None) or (len(_sx_merge_track) < 3):
    raise RuntimeError(
        "Merged trajectory `_sx_merge_track` is not available. "
        "Run the reference-extraction cell first."
    )

_sx_phase_src = np.asarray(_sx_merge_track, dtype=float)
_sy_phase_src = np.asarray(_sy_merge_track, dtype=float)

_phase_pts0 = np.column_stack([_sx_phase_src, _sy_phase_src])

# Remove duplicated closure point if present.
if np.linalg.norm(_phase_pts0[0] - _phase_pts0[-1]) < 1e-12:
    _phase_pts0 = _phase_pts0[:-1]

# Closed arc-length parametrization.
_phase_closed = np.vstack([_phase_pts0, _phase_pts0[0]])
_phase_dxy = np.diff(_phase_closed, axis=0)
_phase_ds = np.hypot(_phase_dxy[:, 0], _phase_dxy[:, 1])
_phase_s = np.concatenate([[0.0], np.cumsum(_phase_ds)])
_phase_L = float(_phase_s[-1])
if _phase_L <= 1e-12:
    raise RuntimeError('Phase source has near-zero arc length.')

# Uniform phase lattice with endpoint=False (last point is one step before wrap).
_phase_su = np.linspace(0.0, _phase_L, N_SUBSAMPLE, endpoint=False)
_sx_phase = np.interp(_phase_su, _phase_s, _phase_closed[:, 0])
_sy_phase = np.interp(_phase_su, _phase_s, _phase_closed[:, 1])

# Enforce phase=0 at the leftmost sampled phase point.
_i_left = int(np.argmin(_sx_phase))
_sx_ref = np.roll(_sx_phase, -_i_left)
_sy_ref = np.roll(_sy_phase, -_i_left)

_ref_pts = np.column_stack([_sx_ref, _sy_ref])
_raw_pts_full = np.column_stack([_ax_raw, _ay_raw])
_all_d = np.linalg.norm(_raw_pts_full[None, :, :] - _ref_pts[:, None, :], axis=2)

_matched_idx = np.argmin(_all_d, axis=1)
_matched_dist = _all_d[np.arange(N_SUBSAMPLE), _matched_idx]

# Optional uniqueness across matched raw points (kept distance-optimal greedily).
if FORCE_UNIQUE:
    _order = np.argsort(_matched_dist)
    _taken = np.zeros(len(_ax_raw), dtype=bool)
    _idx_u = np.empty_like(_matched_idx)
    for _ri in _order:
        _j = int(_matched_idx[_ri])
        if not _taken[_j]:
            _idx_u[_ri] = _j
            _taken[_j] = True
        else:
            _cand = np.argsort(_all_d[_ri])
            for _jj in _cand:
                if not _taken[_jj]:
                    _idx_u[_ri] = int(_jj)
                    _taken[_jj] = True
                    break
    _matched_idx = _idx_u
    _matched_dist = np.linalg.norm(_raw_pts_full[_matched_idx] - _ref_pts, axis=1)

# Keep these arrays explicitly phase-ordered for downstream shape fitting.
_phase_order = np.arange(N_SUBSAMPLE)
_phase_frac = _phase_order / float(N_SUBSAMPLE)

_ax_sub = _ax_raw[_matched_idx]
_ay_sub = _ay_raw[_matched_idx]
_n_unique = len(np.unique(_matched_idx))

c0_fit = _c0_dec[_matched_idx]
c90_fit = _c90_dec[_matched_idx]
c45_fit = _c45_dec[_matched_idx]
c135_fit = _c135_dec[_matched_idx]

print('Phase-matching for shape fitting:')
print('  source: merged track')
print(f'  N phase points: {N_SUBSAMPLE}')
print('  phase parameterization: one full 0..2pi turn on the selected merged trajectory')
print(f'  matched unique raw points: {_n_unique}/{N_SUBSAMPLE}')
print(f'  distance: mean={np.mean(_matched_dist):.4f}, max={np.max(_matched_dist):.4f}')
print(f'  fit arrays rebuilt: {len(c0_fit)} points (phase-ordered)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
axes[0].scatter(_ax_raw, _ay_raw, s=1, alpha=0.07, c='grey', rasterized=True, label='fit-window points')
axes[0].plot(_sx_ref, _sy_ref, '-', c='tab:red', lw=1.5, alpha=0.95, label='phase lattice (ordered)')
axes[0].scatter(_sx_ref, _sy_ref, s=12, c=np.linspace(0, 1, N_SUBSAMPLE), cmap='hsv', alpha=0.9, label='phase samples')
axes[0].scatter(_sx_ref[0], _sy_ref[0], s=36, c='k', zorder=5, label='phase 0 (leftmost)')
axes[0].set_title('Merged-track phase lattice')
axes[0].legend(fontsize=8)

axes[1].scatter(_ax_raw, _ay_raw, s=1, alpha=0.07, c='grey', rasterized=True, label='fit-window points')
axes[1].scatter(_ax_sub, _ay_sub, s=16, c=_phase_frac, cmap='hsv', alpha=0.95, label='nearest matched points')
axes[1].plot(_sx_ref, _sy_ref, '-', c='tab:red', lw=1.0, alpha=0.55, label='phase lattice')
axes[1].set_title('Matched points (ordered by merged-track phase)')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('ax')
    ax.set_ylabel('ay')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Phase-matching for shape fitting:
  source: merged track
  N phase points: 90
  phase parameterization: one full 0..2pi turn on the selected merged trajectory
  matched unique raw points: 90/90
  distance: mean=0.0009, max=0.0027
  fit arrays rebuilt: 90 points (phase-ordered)


In [9]:
# ── Channel correction (phase-ordered fit segment) ────────────────────────────
# c0_fit, c90_fit, c45_fit, c135_fit are built in Section 8 from merged-track phase matching.

# Stack-order index map [c0, c90, c45, c135] -> positions in the [c90, c45, c135, c0] stack.
# Used here and re-used by all downstream cells operating on `tcor_*` stacks.
_idx_fit = get_pol_ind(["0", "90", "45", "135"])

if not use_tmatrix:
    matcor = np.eye(4)
    print("T-matrix correction disabled — using identity matrix.")

if a_manual is not None:
    a = list(a_manual)
    print(f"Channel correction: using manual a = {[f'{v:.4f}' for v in a]}")
else:
    gain_fit = find_best_coeff_using_mat(c0_fit, c90_fit, c45_fit, c135_fit, matcorb)
    l90, l45, l135 = gain_fit.x
    a = [1.0, 1.0, 1.0, 1.0]
    a[get_pol_ind(["90"])[0]] = float(l90)
    a[get_pol_ind(["45"])[0]] = float(l45)
    a[get_pol_ind(["135"])[0]] = float(l135)
    print(f"Channel correction: computed a = {[f'{v:.4f}' for v in a]}")

print(f"Gain   a = {[f'{v:.4g}' for v in a]}")
if any(v < 0 for v in a):
    print(f"WARNING: negative channel coefficient — {a}")

fit_stack = np.vstack((c90_fit, c45_fit, c135_fit, c0_fit)).astype(float)
for i in range(4):
    fit_stack[i] = a[i] * fit_stack[i]

_neg_per_ch = [int(np.sum(fit_stack[i] < 0)) for i in range(4)]
if any(_neg_per_ch):
    _ch_labels = ["c90", "c45", "c135", "c0"]
    print("WARNING: negative samples after gain in fit segment — "
          + ", ".join(f"{lbl}={n}" for lbl, n in zip(_ch_labels, _neg_per_ch))
          + " (consider masking dropouts upstream).")


tcor_fit_sub = np.dot(matcor, fit_stack)
del fit_stack

print(f"Gain + T-matrix applied to fit segment ({tcor_fit_sub.shape[1]} pts).")

Channel correction: computed a = ['1.0083', '1.2804', '1.0917', '1.0000']
Gain   a = ['1.008', '1.28', '1.092', '1']
Gain + T-matrix applied to fit segment (90 pts).


In [10]:
# ── Fourkas calibration constants ─────────────────────────────────────────────

NA_CALIBRATED = 1.28671   # effective NA from calibration (original NA = 1.3)
nw            = 1.33      # refractive index of immersion medium

# ── Hole-correction toggle ─────────────────────────────────────────────────────
USE_HOLE_CORRECTION = True  # set True to use annular (hole) collection path

# Annular collection NA limits (only used when USE_HOLE_CORRECTION = True)
NA_out = 1.3    # outer NA of collection annulus
NA_in  = 0.38   # inner NA (hole edge)

if USE_HOLE_CORRECTION:
    # ── Hole-corrected coefficients from NA cutoff differences ────────────────
    alpha_out = np.arcsin(NA_out / nw)   # outer half-angle
    alpha_in  = np.arcsin(NA_in  / nw)   # inner half-angle (hole)

    A_out, B_out, C_out = fourkas_ABC(alpha_out)
    A_in,  B_in,  C_in  = fourkas_ABC(alpha_in)

    A_cal = A_out - A_in
    B_cal = B_out - B_in
    C_cal = C_out - C_in

    print(f"Hole correction ON  (NA_in={NA_in}, NA_out={NA_out}, n={nw})")
    print(f"alpha_in={np.degrees(alpha_in):.4f}°  alpha_out={np.degrees(alpha_out):.4f}°")
    print(f"A_out={A_out:.6f}  B_out={B_out:.6f}  C_out={C_out:.6f}")
    print(f"A_in ={A_in:.6f}  B_in ={B_in:.6f}  C_in ={C_in:.6f}")
else:
    # ── Legacy path (full-cone Fourkas) ───────────────────────────────────────
    alpha_calib   = np.arcsin(NA_CALIBRATED / nw)
    A_cal, B_cal, C_cal = fourkas_ABC(alpha_calib)

    print(f"Hole correction OFF — legacy Fourkas path")
    print(f"NA_calibrated = {NA_CALIBRATED}  (original NA = 1.3)")
    print(f"α = {np.degrees(alpha_calib):.4f}°")

R_SAT = C_cal / (A_cal + B_cal)

print(f"A={A_cal:.6f}  B={B_cal:.6f}  C={C_cal:.6f}")
print(f"r_sat = {R_SAT:.5f}")

# Make these available to downstream cells
A, B, C = A_cal, B_cal, C_cal

Hole correction ON  (NA_in=0.38, NA_out=1.3, n=1.33)
alpha_in=16.6015°  alpha_out=77.8075°
A_out=0.114653  B_out=0.025222  C_out=0.129650
A_in =0.000428  B_in =0.009779  C_in =0.010206
A=0.114224  B=0.015443  C=0.119444
r_sat = 0.92116


In [11]:
# Restored from 9b3604e:Fourkas_processing_v4.ipynb cell 25 (2026-10-04).
# ══════════════════════════════════════════════════════════════════════════════
# SHAPE FIT — Arc-length matching with two-stage alternating optimisation
# ══════════════════════════════════════════════════════════════════════════════
# tcor_fit_sub contains the phase-ordered fit segment (gain + T-matrix
# corrected). We use all points directly for the fit.

# ── User-configurable optimisation parameters ────────────────────────────
N_OUTER_ITERATIONS = 4
DE_MAXITER = 200
DE_POPSIZE = 15
DE_TOL     = 1e-6
N_SHAPE_FIT_RESAMPLE = 360  # uniform arc-length samples used in shape_fit_cost

# ── Arc-length helpers ───────────────────────────────────────────────────────
def _cumulative_arclen(ax, ay):
    dx = np.diff(ax)
    dy = np.diff(ay)
    ds = np.sqrt(dx**2 + dy**2)
    s = np.concatenate([[0.0], np.cumsum(ds)])
    total = s[-1]
    if total > 0:
        s /= total
    return s

def _reorder_from_leftmost(ax, ay):
    i0 = int(np.argmin(ax))
    return np.roll(ax, -i0), np.roll(ay, -i0)

def _interp_curve_at_fractions(ax, ay, fracs):
    s = _cumulative_arclen(ax, ay)
    return np.interp(fracs, s, ax), np.interp(fracs, s, ay)

def _resample_uniform_arclen_closed(ax, ay, n):
    """Return n samples uniformly spaced in arc length on the closed curve."""
    ax_c = np.concatenate([ax, ax[:1]])
    ay_c = np.concatenate([ay, ay[:1]])
    s = _cumulative_arclen(ax_c, ay_c)
    fracs = np.linspace(0.0, 1.0, n, endpoint=False)
    return np.interp(fracs, s, ax_c), np.interp(fracs, s, ay_c)

# ── Extract fit-segment channel data ─────────────────────────────────────
_c0_cyc   = tcor_fit_sub[_idx_fit[0]]
_c90_cyc  = tcor_fit_sub[_idx_fit[1]]
_c45_cyc  = tcor_fit_sub[_idx_fit[2]]
_c135_cyc = tcor_fit_sub[_idx_fit[3]]

print(f"Fit channel data for shape fitting: {len(_c0_cyc)} points")
print("Model phase domain: fourkas_template uses psi in [0, 2pi) for each evaluation (N_model=720 in fit).")

# ── Cost function — arc-length matched residuals ─────────────────────────
def shape_fit_cost(params, cyc_c0, cyc_c90, cyc_c45, cyc_c135, A, B, C):
    Theta_ax, Phi_ax, Lam, dc, fa, fb = params
    da = fa * dc
    db = fb * dc
    b0   = (dc + da) / 2
    b90  = (dc - da) / 2
    b45  = (dc + db) / 2
    b135 = (dc - db) / 2

    c0_adj   = cyc_c0 - b0
    c90_adj  = cyc_c90 - b90
    c45_adj  = cyc_c45 - b45
    c135_adj = cyc_c135 - b135

    valid = (c0_adj > 0) & (c90_adj > 0) & (c45_adj > 0) & (c135_adj > 0)
    if valid.sum() < 20:
        return 1e6

    ax_d = (c0_adj[valid] - c90_adj[valid]) / (c0_adj[valid] + c90_adj[valid])
    ay_d = (c45_adj[valid] - c135_adj[valid]) / (c45_adj[valid] + c135_adj[valid])

    ax_d_r, ay_d_r = _reorder_from_leftmost(ax_d, ay_d)

    ax_m, ay_m = fourkas_template(Theta_ax, Phi_ax, Lam, A, B, C, N_model=720)
    if len(ax_m) < 5:
        return 1e6
    ax_m_r, ay_m_r = _reorder_from_leftmost(ax_m, ay_m)

    # Resample BOTH data and model to a common uniform arc-length grid so
    # the comparison is parameterisation-consistent (both uniform in arc
    # length, anchored at the leftmost point).
    n_match = max(int(N_SHAPE_FIT_RESAMPLE), len(ax_d_r))
    ax_d_u, ay_d_u = _resample_uniform_arclen_closed(ax_d_r, ay_d_r, n_match)
    ax_m_u, ay_m_u = _resample_uniform_arclen_closed(ax_m_r, ay_m_r, n_match)
    cost_fwd = np.mean((ax_d_u - ax_m_u)**2 + (ay_d_u - ay_m_u)**2)

    ax_m_rev_r, ay_m_rev_r = _reorder_from_leftmost(ax_m[::-1], ay_m[::-1])
    ax_m_rev_u, ay_m_rev_u = _resample_uniform_arclen_closed(ax_m_rev_r, ay_m_rev_r, n_match)
    cost_rev = np.mean((ax_d_u - ax_m_rev_u)**2 + (ay_d_u - ay_m_rev_u)**2)

    return min(cost_fwd, cost_rev)

# ── δc upper bound ───────────────────────────────────────────────────────
dc_upper = max(0.0, min(float(np.min(_c0_cyc) + np.min(_c90_cyc)),
                        float(np.min(_c45_cyc) + np.min(_c135_cyc))))
print(f"δc upper bound: {dc_upper:.4f}")

# ── Parameter bounds ─────────────────────────────────────────────────────
_bounds_geom = [(0.0, 90.0), (-180.0, 180.0), (0.0, 90.0)]
_bounds_bg   = [(0.0, dc_upper), (-1.0, 1.0), (-1.0, 1.0)]  # fa, fb in [-1, 1] keeps b_k >= 0.
_bounds_all  = _bounds_geom + _bounds_bg

def _geom_cost(geom_params, dc, fa, fb, ec0, ec90, ec45, ec135, A, B, C):
    return shape_fit_cost([*geom_params, dc, fa, fb], ec0, ec90, ec45, ec135, A, B, C)

def _bg_cost(bg_params, Theta, Phi, Lam, ec0, ec90, ec45, ec135, A, B, C):
    return shape_fit_cost([Theta, Phi, Lam, *bg_params], ec0, ec90, ec45, ec135, A, B, C)

# ── Alternating optimization with diagnostics ────────────────────────────
_cur_dc, _cur_fa, _cur_fb = 0.0, 0.0, 0.0
_cur_Th, _cur_Ph, _cur_Lm = 45.0, 0.0, 20.0

print(f"\nRunning {N_OUTER_ITERATIONS}-iteration alternating optimization:")
print("  Stage 1: DE on (Θ, Φ, Λ) with background fixed")
print("  Stage 2: DE on (δc, fa, fb) with geometry fixed\n")

_diag_axes = plt.subplots(2, N_OUTER_ITERATIONS, figsize=(5 * N_OUTER_ITERATIONS, 10), dpi=100)[1]

for _outer in range(N_OUTER_ITERATIONS):
    print(f"{'='*70}")
    print(f"Outer iteration {_outer+1}/{N_OUTER_ITERATIONS}")
    print(f"{'='*70}")

    print(f"  Stage 1: fit (Θ, Φ, Λ) | fixed δc={_cur_dc:.4f}, fa={_cur_fa:.4f}, fb={_cur_fb:.4f}")
    _res_geom = differential_evolution(
        _geom_cost,
        bounds=_bounds_geom,
        args=(_cur_dc, _cur_fa, _cur_fb,
              _c0_cyc, _c90_cyc, _c45_cyc, _c135_cyc,
              A_cal, B_cal, C_cal),
        maxiter=DE_MAXITER, popsize=DE_POPSIZE, tol=DE_TOL,
        seed=42 + _outer, polish=True, disp=False, workers=1,
    )
    _cur_Th, _cur_Ph, _cur_Lm = _res_geom.x
    print(f"    → Θ={_cur_Th:.2f}°  Φ={_cur_Ph:.2f}°  Λ={_cur_Lm:.2f}°  cost={_res_geom.fun:.4g}")

    _da_d = _cur_fa * _cur_dc
    _db_d = _cur_fb * _cur_dc
    _c0_d   = _c0_cyc   - (_cur_dc + _da_d) / 2
    _c90_d  = _c90_cyc  - (_cur_dc - _da_d) / 2
    _c45_d  = _c45_cyc  - (_cur_dc + _db_d) / 2
    _c135_d = _c135_cyc - (_cur_dc - _db_d) / 2
    _v_d = (_c0_d > 0) & (_c90_d > 0) & (_c45_d > 0) & (_c135_d > 0)
    _ax_d = (_c0_d[_v_d] - _c90_d[_v_d]) / (_c0_d[_v_d] + _c90_d[_v_d])
    _ay_d = (_c45_d[_v_d] - _c135_d[_v_d]) / (_c45_d[_v_d] + _c135_d[_v_d])

    _ax_mr, _ay_mr = fourkas_template(_cur_Th, _cur_Ph, _cur_Lm, A_cal, B_cal, C_cal, N_model=720)

    _ax_d_r, _ay_d_r = _reorder_from_leftmost(_ax_d, _ay_d)
    _s_d_diag = _cumulative_arclen(_ax_d_r, _ay_d_r)
    _ax_m_r, _ay_m_r = _reorder_from_leftmost(_ax_mr, _ay_mr)
    _s_m_diag = _cumulative_arclen(_ax_m_r, _ay_m_r)

    _a1 = _diag_axes[0, _outer]
    _a1.scatter(_ax_d_r, _ay_d_r, s=4, c=_s_d_diag, cmap='hsv', alpha=0.6, label=f'data ({len(_ax_d)} pts)')
    _a1.scatter(_ax_m_r, _ay_m_r, s=2, c=_s_m_diag, cmap='hsv', alpha=0.4, marker='x', label=f'model ({len(_ax_mr)} pts)')
    _a1.set_title(
        f"Iter {_outer+1} — after geometry\n"
        f"Θ={_cur_Th:.1f}° Φ={_cur_Ph:.1f}° Λ={_cur_Lm:.1f}°\n"
        f"δc={_cur_dc:.4f} fa={_cur_fa:.3f} fb={_cur_fb:.3f}",
        fontsize=9,
    )
    _a1.set_xlim(-1, 1)
    _a1.set_ylim(-1, 1)
    _a1.set_aspect('equal')
    _a1.axhline(0, color='k', lw=0.3)
    _a1.axvline(0, color='k', lw=0.3)
    _a1.grid(alpha=0.2)
    _a1.legend(fontsize=6, loc='lower left')

    print(f"  Stage 2: fit (δc, fa, fb) | fixed Θ={_cur_Th:.2f}°, Φ={_cur_Ph:.2f}°, Λ={_cur_Lm:.2f}°")
    _res_bg = differential_evolution(
        _bg_cost,
        bounds=_bounds_bg,
        args=(_cur_Th, _cur_Ph, _cur_Lm,
              _c0_cyc, _c90_cyc, _c45_cyc, _c135_cyc,
              A_cal, B_cal, C_cal),
        maxiter=DE_MAXITER, popsize=DE_POPSIZE, tol=DE_TOL,
        seed=42 + _outer, polish=True, disp=False, workers=1,
    )
    _cur_dc, _cur_fa, _cur_fb = _res_bg.x
    print(f"    → δc={_cur_dc:.4f}  fa={_cur_fa:.4f}  fb={_cur_fb:.4f}  cost={_res_bg.fun:.4g}")

    _da_d2 = _cur_fa * _cur_dc
    _db_d2 = _cur_fb * _cur_dc
    _c0_d2   = _c0_cyc   - (_cur_dc + _da_d2) / 2
    _c90_d2  = _c90_cyc  - (_cur_dc - _da_d2) / 2
    _c45_d2  = _c45_cyc  - (_cur_dc + _db_d2) / 2
    _c135_d2 = _c135_cyc - (_cur_dc - _db_d2) / 2
    _v_d2 = (_c0_d2 > 0) & (_c90_d2 > 0) & (_c45_d2 > 0) & (_c135_d2 > 0)
    _ax_d2 = (_c0_d2[_v_d2] - _c90_d2[_v_d2]) / (_c0_d2[_v_d2] + _c90_d2[_v_d2])
    _ay_d2 = (_c45_d2[_v_d2] - _c135_d2[_v_d2]) / (_c45_d2[_v_d2] + _c135_d2[_v_d2])

    _ax_mr2, _ay_mr2 = fourkas_template(_cur_Th, _cur_Ph, _cur_Lm, A_cal, B_cal, C_cal, N_model=720)

    _ax_d2_r, _ay_d2_r = _reorder_from_leftmost(_ax_d2, _ay_d2)
    _s_d2_diag = _cumulative_arclen(_ax_d2_r, _ay_d2_r)
    _ax_m2_r, _ay_m2_r = _reorder_from_leftmost(_ax_mr2, _ay_mr2)
    _s_m2_diag = _cumulative_arclen(_ax_m2_r, _ay_m2_r)

    _a2 = _diag_axes[1, _outer]
    _a2.scatter(_ax_d2_r, _ay_d2_r, s=4, c=_s_d2_diag, cmap='hsv', alpha=0.6, label=f'data ({len(_ax_d2)} pts)')
    _a2.scatter(_ax_m2_r, _ay_m2_r, s=2, c=_s_m2_diag, cmap='hsv', alpha=0.4, marker='x', label=f'model ({len(_ax_mr2)} pts)')
    _a2.set_title(
        f"Iter {_outer+1} — after background\n"
        f"Θ={_cur_Th:.1f}° Φ={_cur_Ph:.1f}° Λ={_cur_Lm:.1f}°\n"
        f"δc={_cur_dc:.4f} fa={_cur_fa:.3f} fb={_cur_fb:.3f}",
        fontsize=9,
    )
    _a2.set_xlim(-1, 1)
    _a2.set_ylim(-1, 1)
    _a2.set_aspect('equal')
    _a2.axhline(0, color='k', lw=0.3)
    _a2.axvline(0, color='k', lw=0.3)
    _a2.grid(alpha=0.2)
    _a2.legend(fontsize=6, loc='lower left')

plt.tight_layout()
plt.show()

# ── Final joint polish ───────────────────────────────────────────────────
x0 = [_cur_Th, _cur_Ph, _cur_Lm, _cur_dc, _cur_fa, _cur_fb]
shape_fit_res = minimize(
    shape_fit_cost,
    x0,
    args=(_c0_cyc, _c90_cyc, _c45_cyc, _c135_cyc, A_cal, B_cal, C_cal),
    bounds=_bounds_all,
    method='L-BFGS-B',
)

Theta_axis_fitted, Phi_axis_fitted, Lambda_fitted, dc_fitted, fa_fitted, fb_fitted = shape_fit_res.x
print(f"\nFinal fit:")
print(f"Θ_axis = {Theta_axis_fitted:.3f}°")
print(f"Φ_axis = {Phi_axis_fitted:.3f}°")
print(f"Λ      = {Lambda_fitted:.3f}°")
print(f"δc     = {dc_fitted:.6f}")
print(f"fa     = {fa_fitted:.6f}")
print(f"fb     = {fb_fitted:.6f}")
print(f"Cost   = {shape_fit_res.fun:.6g}")

# ── Background decomposition ─────────────────────────────────────────────
da_fitted = fa_fitted * dc_fitted
db_fitted = fb_fitted * dc_fitted
b0_fitted   = (dc_fitted + da_fitted) / 2
b90_fitted  = (dc_fitted - da_fitted) / 2
b45_fitted  = (dc_fitted + db_fitted) / 2
b135_fitted = (dc_fitted - db_fitted) / 2

print("\nDerived channel backgrounds:")
print(f"b0   = {b0_fitted:.6f}")
print(f"b90  = {b90_fitted:.6f}")
print(f"b45  = {b45_fitted:.6f}")
print(f"b135 = {b135_fitted:.6f}")

# ── Numerical Hessian uncertainty estimate ───────────────────────────────
def numerical_hessian(func, x, eps=1e-5):
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n), dtype=float)
    f0 = func(x)
    for i in range(n):
        xi_p = x.copy(); xi_p[i] += eps
        xi_m = x.copy(); xi_m[i] -= eps
        f_ip = func(xi_p)
        f_im = func(xi_m)
        H[i, i] = (f_ip - 2 * f0 + f_im) / (eps ** 2)
        for j in range(i + 1, n):
            x_pp = x.copy(); x_pp[i] += eps; x_pp[j] += eps
            x_pm = x.copy(); x_pm[i] += eps; x_pm[j] -= eps
            x_mp = x.copy(); x_mp[i] -= eps; x_mp[j] += eps
            x_mm = x.copy(); x_mm[i] -= eps; x_mm[j] -= eps
            f_pp = func(x_pp)
            f_pm = func(x_pm)
            f_mp = func(x_mp)
            f_mm = func(x_mm)
            hij = (f_pp - f_pm - f_mp + f_mm) / (4 * eps ** 2)
            H[i, j] = hij
            H[j, i] = hij
    return H

_obj = lambda p: shape_fit_cost(p, _c0_cyc, _c90_cyc, _c45_cyc, _c135_cyc, A_cal, B_cal, C_cal)
H = numerical_hessian(_obj, shape_fit_res.x, eps=1e-5)

try:
    cov = np.linalg.inv(H)
    diag = np.diag(cov)
    uncertainties = np.sqrt(np.abs(diag))
except np.linalg.LinAlgError:
    cov = np.full_like(H, np.nan)
    uncertainties = np.full(6, np.nan)

print("\nHistorical inverse-Hessian curvature diagnostic (NOT calibrated 1-sigma uncertainties):")
print(f"σ(Θ_axis) = {uncertainties[0]:.4g}")
print(f"σ(Φ_axis) = {uncertainties[1]:.4g}")
print(f"σ(Λ)      = {uncertainties[2]:.4g}")
print(f"σ(δc)     = {uncertainties[3]:.4g}")
print(f"σ(fa)     = {uncertainties[4]:.4g}")
print(f"σ(fb)     = {uncertainties[5]:.4g}")

Fit channel data for shape fitting: 90 points
Model phase domain: fourkas_template uses psi in [0, 2pi) for each evaluation (N_model=720 in fit).
δc upper bound: 0.2016

Running 4-iteration alternating optimization:
  Stage 1: DE on (Θ, Φ, Λ) with background fixed
  Stage 2: DE on (δc, fa, fb) with geometry fixed

Outer iteration 1/4
  Stage 1: fit (Θ, Φ, Λ) | fixed δc=0.0000, fa=0.0000, fb=0.0000
    → Θ=61.82°  Φ=62.20°  Λ=72.37°  cost=0.03753
  Stage 2: fit (δc, fa, fb) | fixed Θ=61.82°, Φ=62.20°, Λ=72.37°
    → δc=0.0515  fa=-0.2926  fb=0.0819  cost=0.0254
Outer iteration 2/4
  Stage 1: fit (Θ, Φ, Λ) | fixed δc=0.0515, fa=-0.2926, fb=0.0819
    → Θ=60.53°  Φ=-118.32°  Λ=71.45°  cost=0.02432
  Stage 2: fit (δc, fa, fb) | fixed Θ=60.53°, Φ=-118.32°, Λ=71.45°
    → δc=0.0677  fa=-0.0670  fb=-0.0595  cost=0.02377
Outer iteration 3/4
  Stage 1: fit (Θ, Φ, Λ) | fixed δc=0.0677, fa=-0.0670, fb=-0.0595
    → Θ=58.73°  Φ=-118.65°  Λ=70.95°  cost=0.02321
  Stage 2: fit (δc, fa, fb) | fixed Θ

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# PROCESSED WINDOW — phase mapping + optional trajectory-change guard
# ══════════════════════════════════════════════════════════════════════════════

# Processing window: this is the data that will be processed downstream.
data_t_start = None     # s — processing window start (use None for full raw range)
data_t_end   = None     # s — processing window end (use None for full raw range)

# Optional extra plotting sub-window within the processed data.
plot_t_start = None  # s — figure-only window start (None => no extra clipping)
plot_t_end   = None  # s — figure-only window end (None => no extra clipping)

# ── What this cell produces ────────────────────────────────────────────────────
# The continuity-constrained tracker (below) ALWAYS runs and produces `psi`, the
# continuous rotation-frame phase. `psi` is the primary output and the only thing
# the speed cell (Section 13) consumes — speed is always dψ/dt of `psi`.
#
# Lab-frame orientation angles (theta_lab, phi_lab) are an OPTIONAL extra and are
# completely independent of the speed calculation:
#   RECONSTRUCT_LAB_ANGLES = False -> theta_lab / phi_lab left as NaN (speed-only).
#   RECONSTRUCT_LAB_ANGLES = True  -> reconstruct them from the per-segment axis
#                                     geometry + tracked psi (for orientation plots).
#
# NOTE: the former ANGLE_RECONSTRUCTION_MODE flag (phase / phase_guided /
# direct_fourkas) was removed. Those modes were about LAB-ANGLE reconstruction,
# which is its own useful quantity (the rod's orientation, distinct from its
# rotation speed):
#   - direct_fourkas : invert the Fourkas equations point-by-point from (a_x, a_y)
#                      -> the most direct theta/phi estimate, but with the usual
#                      degeneracy (theta vs 180-theta, phi vs phi+pi).
#   - phase_guided   : same direct Fourkas inversion, degeneracy resolved using the
#                      tracked phase -> the most accurate angle reconstruction.
# They are off this pass's path only because we are focused on SPEED. The angles
# themselves are perfectly good; what is distorted is using d(phi)/dt as a speed
# (lab-frame phi gets compressed/stretched when the axis is tilted -- exactly the
# problem the psi tracker fixes). Ideal end state: speed from the psi projection
# (this cell) + angle from phase_guided Fourkas. Re-add the Fourkas angle
# reconstruction when angle output is wanted; meanwhile RECONSTRUCT_LAB_ANGLES = True
# gives a quick geometry-based theta/phi from psi (no Fourkas inversion).
RECONSTRUCT_LAB_ANGLES = False

# ── Trajectory-change guard (geometry refit; background fixed) ────────────────
# Periodically re-evaluates the kdtree mismatch metric on a short rolling window.
# If the metric exceeds MISMATCH_TRIGGER_FACTOR x baseline, a geometry-only
# refit is launched and a new phase segment is opened. Backgrounds are fixed.
ENABLE_TRAJECTORY_GUARD   = False   # turn off to force a single-segment fit
CHECK_WINDOW_S            = 1.0    # s, length of the rolling check window
RECHECK_PERIOD_S          = 10.0   # s, how often to evaluate the metric
MISMATCH_TRIGGER_FACTOR   = 3.0    # x baseline metric => trigger refit
GEOMETRY_REFIT_MIN_POINTS = 200    # skip refit if window has fewer points

# ── Phase tracking controls (continuity-constrained local-window matcher) ─────
# Each decimated data point (in time order) is matched to the nearest template
# index inside a HARD window of +/- W around the previous match, then refined to a
# continuous (sub-index) phase by projecting onto the local template polyline.
# The window enforces temporal continuity and makes branch jumps near self-
# intersections impossible, provided   max_real_step < W < N/2 (branch separation).
# W is derived from physics:  W = ceil(N * f_max * D / fs * safety).
#   - too small  -> window saturates at high speed (real motion exceeds W, undertracked)
#   - too large  -> noise can hop to the wrong branch near a self-approach
PHASE_DECIMATION        = 10      # raw->phase decimation factor (250 kHz / 10 = 25 kHz)
PHASE_KDTREE_N_MODEL    = 360     # template points per turn (uniform in rotation psi)
PHASE_MAX_ROTATION_HZ   = 2000.0  # physical upper bound on |rotation rate| (BFM)
PHASE_WINDOW_SAFETY     = 2.0     # window = safety x expected max per-sample index step
PHASE_PSI_SMOOTH        = False   # Savgol-smooth tracked psi (OFF: keep raw dynamics)
PHASE_PSI_SAVGOL_WINDOW = 61      # odd window length, in decimated samples
PHASE_PSI_SAVGOL_POLY   = 2       # Savgol polynomial order

from scipy.spatial import cKDTree
from scipy.optimize import minimize
from scipy.signal import savgol_filter

# Apply a and T matrix to full raw timeline
raw_stack = np.vstack((c90_raw, c45_raw, c135_raw, c0_raw)).astype(float)
for i in range(4):
    raw_stack[i] = a[i] * raw_stack[i]
tcor_stack = np.dot(matcor, raw_stack)
del raw_stack

# Subtract fitted per-channel backgrounds
c0   = tcor_stack[_idx_fit[0]] - b0_fitted
c90  = tcor_stack[_idx_fit[1]] - b90_fitted
c45  = tcor_stack[_idx_fit[2]] - b45_fitted
c135 = tcor_stack[_idx_fit[3]] - b135_fitted

# Keep pre-subtraction channels for background/signal diagnostics
c0_nocor   = tcor_stack[_idx_fit[0]]
c90_nocor  = tcor_stack[_idx_fit[1]]
c45_nocor  = tcor_stack[_idx_fit[2]]
c135_nocor = tcor_stack[_idx_fit[3]]

# ── Apply processing mask early (this data is what gets processed) ────────────
_process_mask = np.ones(len(time_s), dtype=bool)
if data_t_start is not None:
    _process_mask &= time_s >= data_t_start
if data_t_end is not None:
    _process_mask &= time_s <= data_t_end
if not np.any(_process_mask):
    raise RuntimeError("Processing window is empty. Adjust data_t_start / data_t_end.")

time_proc = time_s[_process_mask]
c0 = c0[_process_mask]
c90 = c90[_process_mask]
c45 = c45[_process_mask]
c135 = c135[_process_mask]
c0_nocor = c0_nocor[_process_mask]
c90_nocor = c90_nocor[_process_mask]
c45_nocor = c45_nocor[_process_mask]
c135_nocor = c135_nocor[_process_mask]

# Full-resolution anisotropy (used by pre/post comparison cell)
anis_x, anis_y = anisotropy_from_channels(c0, c90, c45, c135)
anis_x_nocor, anis_y_nocor = anisotropy_from_channels(c0_nocor, c90_nocor, c45_nocor, c135_nocor)

# Decimated points for phase tracking and angle reconstruction
_D = int(max(1, PHASE_DECIMATION))
_time_phase = time_proc[::_D]
_ax_phase = anis_x[::_D]
_ay_phase = anis_y[::_D]
_data_pts = np.column_stack([_ax_phase, _ay_phase])

# ── Helpers for mismatch + geometry-only refit ───────────────────────────────
def _window_metric(theta_deg, phi_deg, lam_deg, ax_w, ay_w, A, B, C, n_model=360):
    """Mean squared kdtree distance from window points to the Fourkas template."""
    if len(ax_w) < 4:
        return np.nan
    ax_m, ay_m = fourkas_template(theta_deg, phi_deg, lam_deg, A, B, C, N_model=int(max(60, n_model)))
    tree = cKDTree(np.column_stack([ax_m, ay_m]))
    d, _ = tree.query(np.column_stack([ax_w, ay_w]), k=1)
    return float(np.mean(d * d))

def _fit_geometry_only(theta0, phi0, lam0, ax_w, ay_w, A, B, C, n_model=360):
    """Refit (theta_axis, phi_axis, Lambda) on a window; background is fixed."""
    if len(ax_w) < GEOMETRY_REFIT_MIN_POINTS:
        return theta0, phi0, lam0, np.nan, False

    def _cost(p):
        th, ph, lm = p
        return _window_metric(th, ph, lm, ax_w, ay_w, A, B, C, n_model=n_model)

    res = minimize(
        _cost,
        x0=np.array([theta0, phi0, lam0], dtype=float),
        bounds=[(0.0, 90.0), (-180.0, 180.0), (0.0, 90.0)],
        method="L-BFGS-B",
    )
    if not res.success:
        return theta0, phi0, lam0, np.nan, False
    return float(res.x[0]), float(res.x[1]), float(res.x[2]), float(res.fun), True

# ── Build geometry segments using periodic guard checks ───────────────────────
_M = len(_time_phase)
phase_segment_ranges = []
phase_segment_params = []
phase_refit_events = []

_theta0 = float(Theta_axis_fitted)
_phi0 = float(Phi_axis_fitted)
_lam0 = float(Lambda_fitted)

if (_M < 5) or (not ENABLE_TRAJECTORY_GUARD):
    phase_segment_ranges = [(0, _M)]
    phase_segment_params = [(_theta0, _phi0, _lam0)]
    _baseline_metric = np.nan
else:
    _i_base_s = 0
    _i_base_e = int(np.searchsorted(_time_phase, _time_phase[0] + CHECK_WINDOW_S, side="left"))
    _i_base_e = max(_i_base_e, min(_M, _i_base_s + 5))

    _baseline_metric = _window_metric(
        _theta0, _phi0, _lam0,
        _ax_phase[_i_base_s:_i_base_e], _ay_phase[_i_base_s:_i_base_e],
        A_cal, B_cal, C_cal, n_model=PHASE_KDTREE_N_MODEL,
    )

    _breaks = [0]
    _params = [(_theta0, _phi0, _lam0)]
    _cur_theta, _cur_phi, _cur_lam = _theta0, _phi0, _lam0

    if np.isfinite(_baseline_metric) and _baseline_metric > 0:
        _t_check = float(_time_phase[0] + RECHECK_PERIOD_S)
        _t_end = float(_time_phase[-1])
        while _t_check < _t_end:
            _iw_s = int(np.searchsorted(_time_phase, _t_check, side="left"))
            _iw_e = int(np.searchsorted(_time_phase, _t_check + CHECK_WINDOW_S, side="left"))
            _iw_e = min(_M, max(_iw_e, _iw_s + 5))

            if _iw_s >= _M - 5:
                break

            _metric = _window_metric(
                _cur_theta, _cur_phi, _cur_lam,
                _ax_phase[_iw_s:_iw_e], _ay_phase[_iw_s:_iw_e],
                A_cal, B_cal, C_cal, n_model=PHASE_KDTREE_N_MODEL,
            )

            if np.isfinite(_metric) and (_metric > MISMATCH_TRIGGER_FACTOR * _baseline_metric):
                _new_theta, _new_phi, _new_lam, _new_cost, _ok = _fit_geometry_only(
                    _cur_theta, _cur_phi, _cur_lam,
                    _ax_phase[_iw_s:_iw_e], _ay_phase[_iw_s:_iw_e],
                    A_cal, B_cal, C_cal, n_model=PHASE_KDTREE_N_MODEL,
                )
                if _ok:
                    if _iw_s > _breaks[-1] + 2:
                        _breaks.append(_iw_s)
                        _params.append((_new_theta, _new_phi, _new_lam))
                    phase_refit_events.append({
                        "t": float(_time_phase[_iw_s]),
                        "idx": int(_iw_s),
                        "metric_before": float(_metric),
                        "baseline_metric_before": float(_baseline_metric),
                        "theta_old": float(_cur_theta),
                        "phi_old": float(_cur_phi),
                        "lam_old": float(_cur_lam),
                        "theta_new": float(_new_theta),
                        "phi_new": float(_new_phi),
                        "lam_new": float(_new_lam),
                        "fit_cost": float(_new_cost),
                    })

                    _rb_s = _iw_s
                    _rb_e = int(np.searchsorted(_time_phase, _time_phase[_rb_s] + CHECK_WINDOW_S, side="left"))
                    _rb_e = min(_M, max(_rb_e, _rb_s + 5))
                    _baseline_new = _window_metric(
                        _new_theta, _new_phi, _new_lam,
                        _ax_phase[_rb_s:_rb_e], _ay_phase[_rb_s:_rb_e],
                        A_cal, B_cal, C_cal, n_model=PHASE_KDTREE_N_MODEL,
                    )
                    if np.isfinite(_baseline_new) and (_baseline_new > 0):
                        _baseline_metric = float(_baseline_new)
                    _cur_theta, _cur_phi, _cur_lam = _new_theta, _new_phi, _new_lam

            _t_check += RECHECK_PERIOD_S

    _breaks.append(_M)
    for _si in range(len(_params)):
        phase_segment_ranges.append((_breaks[_si], _breaks[_si + 1]))
        phase_segment_params.append(_params[_si])

# ── Segment-wise local-window phase tracking (continuity-constrained) ─────────
# Template resolution (uniform in psi). Sub-index projection makes the recovered
# psi continuous, so a moderate N is enough; finer N mainly sharpens the polyline.
_N = int(max(60, PHASE_KDTREE_N_MODEL))

# Physically-derived continuity window (template-index units per decimated sample).
# A real rotation of at most PHASE_MAX_ROTATION_HZ advances at most N*f_max/fs_eff
# indices per sample; we allow PHASE_WINDOW_SAFETY x that, clamped below N/3 so it
# can never reach the antipodal branch (~N/2 away) at a near-vertical self-overlap.
_fs_eff = float(freq) / float(_D)
_max_step_expected = _N * float(PHASE_MAX_ROTATION_HZ) / _fs_eff
_W = int(np.ceil(_max_step_expected * float(PHASE_WINDOW_SAFETY)))
_W = int(np.clip(_W, 3, max(3, _N // 3)))
_offsets = np.arange(-_W, _W + 1)

psi = np.empty(_M, dtype=float)
_idx_path = np.empty(_M, dtype=int)
_step_idx = np.zeros(_M, dtype=float)
theta_axis_used = np.empty(_M, dtype=float)
phi_axis_used = np.empty(_M, dtype=float)
lambda_used = np.empty(_M, dtype=float)

def _build_model(theta_deg, phi_deg, lam_deg, n_model):
    ax_m, ay_m = fourkas_template(theta_deg, phi_deg, lam_deg, A_cal, B_cal, C_cal, N_model=n_model)
    return np.column_stack([ax_m, ay_m])

def _subindex_project(pt, model_pts, j, n):
    """Continuous fractional index near integer index j: project pt onto the two
    adjacent template segments [j-1, j] and [j, j+1] (circular) and keep the
    closer foot point. Returns a fractional index whose integer part is j-1 or j."""
    best_frac = float(j)
    best_d2 = np.inf
    for j0 in (j - 1, j):
        a0 = model_pts[j0 % n]
        b0v = model_pts[(j0 + 1) % n]
        abx = b0v[0] - a0[0]
        aby = b0v[1] - a0[1]
        denom = abx * abx + aby * aby
        if denom <= 1e-30:
            continue
        t = ((pt[0] - a0[0]) * abx + (pt[1] - a0[1]) * aby) / denom
        if t < 0.0:
            t = 0.0
        elif t > 1.0:
            t = 1.0
        fx = a0[0] + t * abx - pt[0]
        fy = a0[1] + t * aby - pt[1]
        d2 = fx * fx + fy * fy
        if d2 < best_d2:
            best_d2 = d2
            best_frac = j0 + t
    return best_frac

_cum = 0.0           # continuous (unwrapped) template index
_have_prev = False
_n_saturated = 0     # samples whose best integer index hit the window edge (undertracked)

for (_s, _e), (_th_seg, _ph_seg, _lam_seg) in zip(phase_segment_ranges, phase_segment_params):
    if _e - _s <= 0:
        continue

    theta_axis_used[_s:_e] = _th_seg
    phi_axis_used[_s:_e] = _ph_seg
    lambda_used[_s:_e] = _lam_seg

    _model_pts = _build_model(_th_seg, _ph_seg, _lam_seg, _N)

    for _ii in range(_e - _s):
        _g = _s + _ii
        _pt = _data_pts[_g]

        if (not _have_prev) or (_ii == 0):
            # Seed (first point) or re-seed at a geometry-segment boundary: pick the
            # globally nearest template index, then preserve the running revolution
            # count so psi stays continuous across the boundary.
            _d2_all = (_model_pts[:, 0] - _pt[0]) ** 2 + (_model_pts[:, 1] - _pt[1]) ** 2
            _j = int(np.argmin(_d2_all))
            _frac = _subindex_project(_pt, _model_pts, _j, _N)
            if not _have_prev:
                _cum = _frac
                _have_prev = True
            else:
                _cum = _frac + _N * np.round((_cum - _frac) / _N)
            _idx_path[_g] = int(_j) % _N
            _step_idx[_g] = 0.0
            psi[_g] = (2.0 * np.pi / _N) * _cum
            continue

        # Restrict candidates to +/- W around the current position (hard window).
        _center = int(np.round(_cum)) % _N
        _cands = (_center + _offsets) % _N
        _mc = _model_pts[_cands]
        _d2 = (_mc[:, 0] - _pt[0]) ** 2 + (_mc[:, 1] - _pt[1]) ** 2
        _j = int(_cands[int(np.argmin(_d2))])

        # Flag undertracking: best integer index sitting on the window edge means
        # the true step may exceed W (raise PHASE_MAX_ROTATION_HZ or lower decimation).
        _jrel = ((_j - _center + _N // 2) % _N) - _N // 2
        if abs(_jrel) >= _W:
            _n_saturated += 1

        # Continuous sub-index refinement, then shortest-path signed step.
        _frac = _subindex_project(_pt, _model_pts, _j, _N)
        _prev_frac = _cum % _N
        _step = (_frac - _prev_frac + _N / 2.0) % _N - _N / 2.0
        # Hard clamp (W+1 allows the sub-index spill past the integer window edge).
        if _step > _W + 1.0:
            _step = float(_W + 1.0)
        elif _step < -(_W + 1.0):
            _step = float(-(_W + 1.0))

        _cum += _step
        _idx_path[_g] = int(_j) % _N
        _step_idx[_g] = _step
        psi[_g] = (2.0 * np.pi / _N) * _cum

if PHASE_PSI_SMOOTH and len(psi) >= 7:
    _w = int(max(7, PHASE_PSI_SAVGOL_WINDOW))
    if _w % 2 == 0:
        _w += 1
    _w = min(_w, len(psi) - (1 - len(psi) % 2))
    if _w >= 7 and _w > PHASE_PSI_SAVGOL_POLY:
        psi = savgol_filter(psi, _w, int(PHASE_PSI_SAVGOL_POLY), mode="interp")

# ── Optional lab-frame orientation angles (psi is always the primary output) ──
# Independent of speed. When RECONSTRUCT_LAB_ANGLES is True, rebuild the rod
# direction per geometry segment from (axis k-hat, basis e1/e2, cone half-angle
# Lambda) + tracked psi, then read off lab-frame theta/phi. Otherwise leave NaN.
# (This is the geometry-from-psi route. The most ACCURATE angle would instead come
# from a direct/phase-guided Fourkas inversion -- see the NOTE at the top -- which
# is intentionally left out of this speed-focused pass.)
def _basis_from_axis(theta_deg, phi_deg):
    th = np.radians(theta_deg)
    ph = np.radians(phi_deg)
    kx = np.sin(th) * np.cos(ph)
    ky = np.sin(th) * np.sin(ph)
    kz = np.cos(th)
    khat = np.array([kx, ky, kz])
    if abs(kz) < 0.9999:
        e1 = np.array([0.0, 0.0, 1.0]) - kz * khat
        e1 /= np.linalg.norm(e1)
    else:
        e1 = np.array([1.0, 0.0, 0.0])
    e2 = np.cross(khat, e1)
    return khat, e1, e2

if RECONSTRUCT_LAB_ANGLES:
    theta_lab = np.empty(_M, dtype=float)
    phi_lab = np.empty(_M, dtype=float)
    for (_s, _e), (_th_seg, _ph_seg, _lam_seg) in zip(phase_segment_ranges, phase_segment_params):
        if _e - _s <= 0:
            continue
        _khat, _e1, _e2 = _basis_from_axis(_th_seg, _ph_seg)
        _psi_seg = psi[_s:_e]
        _lam_seg_rad = np.radians(_lam_seg)
        _d_seg = (
            np.cos(_lam_seg_rad) * _khat[None, :]
            + np.sin(_lam_seg_rad)
            * (np.cos(_psi_seg)[:, None] * _e1[None, :] + np.sin(_psi_seg)[:, None] * _e2[None, :])
        )
        theta_lab[_s:_e] = np.degrees(np.arccos(np.clip(_d_seg[:, 2], -1.0, 1.0)))
        phi_lab[_s:_e] = np.arctan2(_d_seg[:, 1], _d_seg[:, 0])
    phi_lab = np.unwrap(phi_lab)
else:
    theta_lab = np.full(_M, np.nan, dtype=float)
    phi_lab = np.full(_M, np.nan, dtype=float)

# Mask for section-11 figures (on processed timeline)
_mask = np.ones(len(time_proc), dtype=bool)
if plot_t_start is not None:
    _mask &= time_proc >= plot_t_start
if plot_t_end is not None:
    _mask &= time_proc <= plot_t_end

# ── Background-to-signal diagnostics ──────────────────────────────────────────
def _bg_ratio_stats(b_val, c_arr):
    c_pos = c_arr[c_arr > 0]
    if len(c_pos) == 0:
        return {"representative": None, "range": [None, None]}
    r_rep = abs(b_val) / float(np.median(c_pos))
    r_min = abs(b_val) / float(np.max(c_pos))
    r_max = abs(b_val) / float(np.min(c_pos))
    return {"representative": r_rep, "range": [r_min, r_max]}

_bg_ratio_c0 = _bg_ratio_stats(b0_fitted, c0_nocor)
_bg_ratio_c90 = _bg_ratio_stats(b90_fitted, c90_nocor)
_bg_ratio_c45 = _bg_ratio_stats(b45_fitted, c45_nocor)
_bg_ratio_c135 = _bg_ratio_stats(b135_fitted, c135_nocor)

print(f"Raw points available: {len(time_s):,}")
print(f"Processed points after window: {len(time_proc):,} (window={data_t_start}..{data_t_end} s)")
print(f"Phase-tracking points used: {len(_time_phase):,} (decimation={_D}x, fs_eff={_fs_eff:.1f} Hz)")
print(f"Template phase points (one turn): {_N}")
print(f"Continuity window: +/-{_W} idx  (expected max step ~{_max_step_expected:.1f} idx @ {PHASE_MAX_ROTATION_HZ:.0f} Hz; branch sep ~{_N // 2})")
_max_obs_step = float(np.max(np.abs(_step_idx))) if _M > 0 else 0.0
_sat_pct = 100.0 * _n_saturated / max(1, _M)
print(f"Observed max |step|: {_max_obs_step:.2f} idx; window-saturated samples: {_n_saturated} ({_sat_pct:.2f}%)")
if _W >= _N // 2:
    print("  WARNING: window reaches the antipodal branch (W >= N/2); branch jumps possible. Lower PHASE_MAX_ROTATION_HZ/safety or raise N.")
if _sat_pct > 1.0:
    print("  WARNING: >1% of samples saturate the window; real motion may exceed it. Raise PHASE_MAX_ROTATION_HZ or lower PHASE_DECIMATION.")
print(f"Generated phase samples: {len(psi):,}")
print(f"psi (rotation-frame): min={np.nanmin(psi):.4f}  max={np.nanmax(psi):.4f} rad")
print(f"RECONSTRUCT_LAB_ANGLES={RECONSTRUCT_LAB_ANGLES}")
if RECONSTRUCT_LAB_ANGLES:
    print(f"theta_lab: min={np.nanmin(theta_lab):.2f} deg  mean={np.nanmean(theta_lab):.2f} deg  max={np.nanmax(theta_lab):.2f} deg")
    print(f"phi_lab:   min={np.nanmin(phi_lab):.4f}  max={np.nanmax(phi_lab):.4f} rad")
else:
    print("theta_lab / phi_lab: not reconstructed (speed-only); psi is the primary output.")
print(f"Trajectory guard: {'ON' if ENABLE_TRAJECTORY_GUARD else 'OFF'}")
if ENABLE_TRAJECTORY_GUARD:
    print(f"  baseline mismatch metric={_baseline_metric:.6g}")
    print(f"  trigger factor={MISMATCH_TRIGGER_FACTOR:.3f}, check_window={CHECK_WINDOW_S}s, recheck_period={RECHECK_PERIOD_S}s")
    print(f"  geometry segments={len(phase_segment_ranges)}  refits={len(phase_refit_events)}")
    if len(phase_refit_events) > 0:
        for _ev in phase_refit_events:
            print(
                f"    refit @ t={_ev['t']:.3f}s | metric={_ev['metric_before']:.6g} "
                f"({(_ev['metric_before']/_ev['baseline_metric_before']):.2f}x baseline) | "
                f"Theta: {_ev['theta_old']:.2f}->{_ev['theta_new']:.2f}, "
                f"Phi: {_ev['phi_old']:.2f}->{_ev['phi_new']:.2f}, "
                f"Lambda: {_ev['lam_old']:.2f}->{_ev['lam_new']:.2f}"
            )
        print("  NOTE: baseline is reset after each successful refit on a new 1 s window.")
print("Background/signal ratios:")
print("  representative = |b|/median(c), range = [|b|/max(c), |b|/min(c)")
print(f"  c0:   rep={_bg_ratio_c0['representative']:.5f}  range={_bg_ratio_c0['range']}")
print(f"  c90:  rep={_bg_ratio_c90['representative']:.5f}  range={_bg_ratio_c90['range']}")
print(f"  c45:  rep={_bg_ratio_c45['representative']:.5f}  range={_bg_ratio_c45['range']}")
print(f"  c135: rep={_bg_ratio_c135['representative']:.5f}  range={_bg_ratio_c135['range']}")

Raw points available: 10,000,000
Processed points after window: 10,000,000 (window=None..None s)
Phase-tracking points used: 1,000,000 (decimation=10x, fs_eff=25000.0 Hz)
Template phase points (one turn): 360
Continuity window: +/-58 idx  (expected max step ~28.8 idx @ 2000 Hz; branch sep ~180)
Observed max |step|: 59.00 idx; window-saturated samples: 6221 (0.62%)
Generated phase samples: 1,000,000
psi (rotation-frame): min=-68380.5227  max=2.7664 rad
RECONSTRUCT_LAB_ANGLES=False
theta_lab / phi_lab: not reconstructed (speed-only); psi is the primary output.
Trajectory guard: OFF
Background/signal ratios:
  representative = |b|/median(c), range = [|b|/max(c), |b|/min(c)
  c0:   rep=0.16819  range=[np.float64(0.08504213248624336), np.float64(0.6865313275065472)]
  c90:  rep=0.23505  range=[np.float64(0.06137069527978576), np.float64(0.6753173773560801)]
  c45:  rep=0.20375  range=[np.float64(0.05414661054689538), np.float64(0.5476710790550505)]
  c135: rep=0.17925  range=[np.float64(0.0

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — shared analysis toolkit  (define once, reused by every section below)
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
try:
    import allantools
except Exception:
    allantools = None

# decimated phase grid (aligned to psi) ----------------------------------------
_TPH   = time_proc[::PHASE_DECIMATION][: len(psi)]
_DTP   = float(np.median(np.diff(_TPH)))
_FS_EFF = 1.0 / _DTP

# output folder for saved figures (results mirror of the source path) -----
try:
    _output_dir
except NameError:
    _rel = Path(tdms_path).with_suffix("")
    try:
        _rel = _rel.relative_to(Path.cwd())
    except Exception:
        _rel = Path(*_rel.parts[-3:])
    _output_dir = Path("results") / _rel
_output_dir = Path(_output_dir); _output_dir.mkdir(parents=True, exist_ok=True)


def speed_at_dt(dt, t0=None, t1=None, overlap=0.0):
    """Windowed linear-slope speed of psi over [t0,t1] -> (t_centres, speed_Hz)."""
    npw = max(5, int(round(dt / _DTP)))
    stp = max(1, int(round(npw * (1.0 - overlap))))
    sel = np.ones(len(psi), bool)
    if t0 is not None: sel &= _TPH >= t0
    if t1 is not None: sel &= _TPH <= t1
    p = psi[sel]; t = _TPH[sel]
    if len(p) < npw:
        return np.array([]), np.array([])
    n = 1 + (len(p) - npw) // stp
    tc = np.empty(n); v = np.empty(n)
    for w in range(n):
        a = w * stp; b = a + npw; tt = t[a:b] - t[a]
        v[w]  = np.polyfit(tt, p[a:b], 1)[0] / (2.0 * np.pi)
        tc[w] = 0.5 * (t[a] + t[b - 1])
    return tc, v


# --- rotating vs stopped region detection -------------------------------------
ROT_DT, ROT_THRESH, ROT_MINLEN = 0.2, 25.0, 3.0   # window, |speed| Hz cut, min length s

def _morph(mask, n):
    m = mask.copy()
    for fill in (False, True):                     # close gaps then drop short runs
        i = 0
        while i < len(m):
            if m[i] == fill:
                j = i
                while j < len(m) and m[j] == fill:
                    j += 1
                short = (j - i) < n
                if fill is False and short and i > 0 and j < len(m):
                    m[i:j] = True
                if fill is True and short:
                    m[i:j] = False
                i = j
            else:
                i += 1
    return m

def _runs(mask, tc):
    out = []; i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            out.append((float(tc[i]), float(tc[j - 1]))); i = j
        else:
            i += 1
    return out

def detect_regions(thresh=ROT_THRESH, dt=ROT_DT, minlen=ROT_MINLEN):
    tc, v = speed_at_dt(dt, overlap=0.0)
    rot = _morph(np.abs(v) > thresh, max(1, int(round(minlen / dt))))
    return _runs(rot, tc), _runs(~rot, tc), tc, v

ROT_REGIONS, STOP_REGIONS, _RM_T, _RM_V = detect_regions()
CLEAN_ROT = max(ROT_REGIONS, key=lambda r: r[1] - r[0]) if ROT_REGIONS else (float(_TPH[0]), float(_TPH[-1]))


def fit_steps(t0=None, t1=None, dt=0.2, penk=3.0, noise=None, overlap=0.0):
    """Penalised L2 changepoint fit of |speed| -> dict(t,v,fit,edges,levels,steps,dwells,snr,sigma_w,dt)."""
    tc, v = speed_at_dt(dt, t0=t0, t1=t1, overlap=overlap)
    v = np.abs(v); n = len(v)
    out = dict(t=tc, v=v, fit=v.copy(), edges=np.array([]), levels=np.array([]),
               steps=np.array([]), dwells=np.array([]), snr=np.array([]),
               sigma_w=np.nan, dt=dt)
    if n < 6:
        return out
    if noise is None:
        d = np.diff(v)
        noise = max(1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2), 1.0)
    cs  = np.concatenate([[0.0], np.cumsum(v)])
    cs2 = np.concatenate([[0.0], np.cumsum(v * v)])
    def sc(i, j):
        m = (cs[j] - cs[i]) / (j - i)
        return (cs2[j] - cs2[i]) - 2 * m * (cs[j] - cs[i]) + (j - i) * m * m
    pen = penk * (noise ** 2) * np.log(n)
    best = np.full(n + 1, np.inf); best[0] = -pen; back = np.zeros(n + 1, int)
    for j in range(1, n + 1):
        bj = np.inf; bk = 0
        for i in range(j):
            c_ = best[i] + sc(i, j) + pen
            if c_ < bj:
                bj = c_; bk = i
        best[j] = bj; back[j] = bk
    bnds = []; j = n
    while j > 0:
        i = back[j]; bnds.append((i, j)); j = i
    bnds = bnds[::-1]
    lv = np.array([(cs[b] - cs[a]) / (b - a) for a, b in bnds])
    fit = np.empty(n)
    for (a, b), l in zip(bnds, lv):
        fit[a:b] = l
    steps = np.diff(lv); sig_w = float(np.std(v - fit))
    out.update(fit=fit, levels=lv, steps=steps, sigma_w=sig_w,
               edges=np.array([tc[a] for a, b in bnds[1:]]),
               dwells=np.array([tc[b - 1] - tc[a] + dt for a, b in bnds]),
               snr=np.abs(steps) / max(sig_w, 1e-6))
    return out


def omega_adev(t0, t1, n=40):
    """Overlapping Allan deviation of the rotation rate on [t0,t1] -> (tau, sigma_Hz)."""
    if allantools is None:
        return np.array([]), np.array([])
    sel = (_TPH >= t0) & (_TPH <= t1)
    pw = psi[sel].astype(float); tw = _TPH[sel]
    if len(pw) < 1000:
        return np.array([]), np.array([])
    taus = np.logspace(np.log10(2.0 / _FS_EFF), np.log10((tw[-1] - tw[0]) / 5.0), n)
    tau, adev, _, _ = allantools.oadev(pw, rate=_FS_EFF, data_type="phase", taus=taus)
    ah = adev / (2.0 * np.pi); ok = np.isfinite(ah) & (ah > 0)
    return tau[ok], ah[ok]


def track_psi_segment(ax_seg, ay_seg, fs=None, theta=None, phi=None, lam=None,
                      n_model=None, max_rot_hz=None, safety=None):
    """Run the continuity-constrained local-window tracker on a SHORT segment of
    FULL-resolution anisotropy (ax_seg, ay_seg) and return continuous psi (rad),
    one value per input sample (NO decimation). This is the same algorithm as the
    Section-12 tracker, applied on demand to a short window so psi can be obtained
    at the native sample rate without rerunning the whole pipeline. Uses the single
    fitted geometry (Theta/Phi/Lambda_axis_fitted) and calibration A/B/C_cal."""
    ax_seg = np.asarray(ax_seg, float); ay_seg = np.asarray(ay_seg, float)
    m = ax_seg.size
    out = np.full(m, np.nan)
    if m == 0:
        return out
    n  = int(max(60, n_model if n_model is not None else PHASE_KDTREE_N_MODEL))
    th = float(Theta_axis_fitted if theta is None else theta)
    ph = float(Phi_axis_fitted   if phi   is None else phi)
    lm = float(Lambda_fitted     if lam   is None else lam)
    fsv = float(freq if fs is None else fs)
    fmx = float(PHASE_MAX_ROTATION_HZ if max_rot_hz is None else max_rot_hz)
    sf  = float(PHASE_WINDOW_SAFETY   if safety     is None else safety)
    ax_m, ay_m = fourkas_template(th, ph, lm, A_cal, B_cal, C_cal, N_model=n)
    model = np.column_stack([ax_m, ay_m])
    W = int(np.clip(int(np.ceil(n * fmx / fsv * sf)), 3, max(3, n // 3)))
    offs = np.arange(-W, W + 1)
    pts = np.column_stack([ax_seg, ay_seg])

    def _sub(pt, j):
        best_frac = float(j); best_d2 = np.inf
        for j0 in (j - 1, j):
            a0 = model[j0 % n]; b0 = model[(j0 + 1) % n]
            abx = b0[0] - a0[0]; aby = b0[1] - a0[1]
            den = abx * abx + aby * aby
            if den <= 1e-30:
                continue
            t = ((pt[0] - a0[0]) * abx + (pt[1] - a0[1]) * aby) / den
            t = 0.0 if t < 0.0 else (1.0 if t > 1.0 else t)
            fx = a0[0] + t * abx - pt[0]; fy = a0[1] + t * aby - pt[1]
            d2 = fx * fx + fy * fy
            if d2 < best_d2:
                best_d2 = d2; best_frac = j0 + t
        return best_frac

    # seed at globally nearest template index
    d2 = (model[:, 0] - pts[0, 0]) ** 2 + (model[:, 1] - pts[0, 1]) ** 2
    cum = _sub(pts[0], int(np.argmin(d2)))
    out[0] = (2.0 * np.pi / n) * cum
    for i in range(1, m):
        pt = pts[i]
        center = int(np.round(cum)) % n
        cands = (center + offs) % n
        mc = model[cands]
        dd = (mc[:, 0] - pt[0]) ** 2 + (mc[:, 1] - pt[1]) ** 2
        j = int(cands[int(np.argmin(dd))])
        frac = _sub(pt, j)
        step = (frac - (cum % n) + n / 2.0) % n - n / 2.0
        if step > W + 1.0:
            step = float(W + 1.0)
        elif step < -(W + 1.0):
            step = float(-(W + 1.0))
        cum += step
        out[i] = (2.0 * np.pi / n) * cum
    return out


print(f"file      : {Path(tdms_path).name}")
print(f"duration  : {_TPH[0]:.1f}–{_TPH[-1]:.1f} s  ({_TPH[-1]-_TPH[0]:.1f} s), "
      f"{len(psi):,} pts @ {_FS_EFF:.0f} Hz")
print(f"rotating  : {[(round(a,1), round(b,1)) for a, b in ROT_REGIONS]}")
print(f"stopped   : {[(round(a,1), round(b,1)) for a, b in STOP_REGIONS]}")
print(f"CLEAN_ROT : {tuple(round(x,1) for x in CLEAN_ROT)}")
print("toolkit ready: speed_at_dt · detect_regions · fit_steps · omega_adev · track_psi_segment")


file      : files3_first40s.tdms
duration  : 0.0–40.0 s  (40.0 s), 1,000,000 pts @ 25000 Hz
rotating  : [(0.1, 39.9)]
stopped   : []
CLEAN_ROT : (0.1, 39.9)
toolkit ready: speed_at_dt · detect_regions · fit_steps · omega_adev · track_psi_segment


In [14]:
# ── Figure 1: Anisotropy shape (pre- and post-correction) ────────────────────
_dec_sc = max(1, len(anis_x[_mask]) // 5000)
_ax_sc  = anis_x[_mask][::_dec_sc]
_ay_sc  = anis_y[_mask][::_dec_sc]
_ax_nc  = anis_x_nocor[_mask][::_dec_sc]
_ay_nc  = anis_y_nocor[_mask][::_dec_sc]

# Pure model template
_ax_model_pure, _ay_model_pure = fourkas_template(
    Theta_axis_fitted, Phi_axis_fitted, Lambda_fitted, A_cal, B_cal, C_cal)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=300)

ax1.scatter(_ax_nc, _ay_nc, s=1, alpha=0.3, c="tab:orange", rasterized=True)
ax1.plot(np.append(_ax_model_pure, _ax_model_pure[0]),
         np.append(_ay_model_pure, _ay_model_pure[0]),
         'r-', lw=2, label='pure model template')
ax1.set_title("Pre-correction (no background subtraction)")
ax1.legend(fontsize=8)

ax2.scatter(_ax_sc, _ay_sc, s=1, alpha=0.3, c="tab:blue", rasterized=True)
ax2.plot(np.append(_ax_model_pure, _ax_model_pure[0]),
         np.append(_ay_model_pure, _ay_model_pure[0]),
         'r-', lw=2, label='pure model template')
ax2.set_title(f"Post-correction\n"
              f"b0={b0_fitted:+.4f}  b90={b90_fitted:+.4f}  "
              f"b45={b45_fitted:+.4f}  b135={b135_fitted:+.4f}")
ax2.legend(fontsize=8)

for _ax in (ax1, ax2):
    _ax.axhline(0, color="k", lw=0.5, zorder=3)
    _ax.axvline(0, color="k", lw=0.5, zorder=3)
    _ax.set_xlabel("anis_x"); _ax.set_ylabel("anis_y")
    _ax.set_xlim(-1, 1); _ax.set_ylim(-1, 1)
    _ax.set_aspect("equal"); _ax.grid(alpha=0.25)

_proc_label = f"processed window [{data_t_start}, {data_t_end}] s"
fig.suptitle(f"{tdms_path.name} — anisotropy ({_proc_label})")
plt.tight_layout()
fig.savefig(_output_dir / "01_anisotropy_shape.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: 01_anisotropy_shape.png")

Saved: 01_anisotropy_shape.png


In [15]:
# ── 13b. Speed time-series + region map ───────────────────────────────────────
# Speed straight from psi via `speed_at_dt` at a sensible averaging time (no more
# 5 ms micro-windows). One coarse trace shows the rotating/stopped structure; a
# finer trace shows the detail. Shaded = auto-detected regions.
SPEED_DT_FINE   = 0.05    # s — detail trace
SPEED_DT_COARSE = 0.20    # s — structure / region map

tf, vf = speed_at_dt(SPEED_DT_FINE)
tcs, vc = speed_at_dt(SPEED_DT_COARSE)

fig, (ax_sp, ax_z) = plt.subplots(2, 1, figsize=(13, 7), dpi=130,
                                  gridspec_kw=dict(height_ratios=[2, 1]))

ax_sp.plot(tf, vf, lw=0.5, color="0.65", label=f"dt={SPEED_DT_FINE*1e3:.0f} ms")
ax_sp.plot(tcs, vc, lw=1.4, color="tab:purple", label=f"dt={SPEED_DT_COARSE*1e3:.0f} ms")
for a, b in ROT_REGIONS:
    ax_sp.axvspan(a, b, color="tab:green", alpha=0.08)
for a, b in STOP_REGIONS:
    ax_sp.axvspan(a, b, color="tab:red", alpha=0.10)
ax_sp.axhline(0, color="k", lw=0.5)
ax_sp.set_ylabel("speed (Hz)")
ax_sp.set_title(f"speed from psi — {Path(tdms_path).name}   "
                f"(green=rotating, red=stopped)")
ax_sp.legend(loc="upper right", fontsize=8); ax_sp.grid(alpha=0.25)

# revolutions integral (monotone winding check) --------------------------------
ax_z.plot(_TPH, psi / (2 * np.pi), lw=0.8, color="tab:blue")
ax_z.set_xlabel("time (s)"); ax_z.set_ylabel("revolutions")
ax_z.set_title("cumulative revolutions (psi / 2π)"); ax_z.grid(alpha=0.25)

plt.tight_layout()
fig.savefig(_output_dir / "02_speed_from_psi.png", dpi=150, bbox_inches="tight")
plt.show()

_vc_rot = np.abs(vc[(vc != 0)])
print(f"saved: {_output_dir / '02_speed_from_psi.png'}")
print(f"|speed| over file: median={np.median(np.abs(vf)):.1f} Hz, "
      f"max={np.max(np.abs(vf)):.1f} Hz")
print(f"rotating regions : {[(round(a,1), round(b,1)) for a, b in ROT_REGIONS]}")


saved: results/files3_first40s/02_speed_from_psi.png
|speed| over file: median=278.6 Hz, max=372.8 Hz
rotating regions : [(0.1, 39.9)]


In [16]:
# ── 13c. Speed histogram + KDE peaks (candidate quantised levels) ──────────────
# Occupancy histogram of |speed| over a region (default: the longest rotating
# stretch CLEAN_ROT; set HIST_REGION=None for the whole file). KDE peaks mark
# candidate speed levels — a multi-stator BFM tends to dwell at discrete rates.
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

HIST_REGION = CLEAN_ROT      # (t0, t1) or None for whole file
HIST_DT     = 0.10           # s — averaging time for the histogram
HIST_KDE_BW = 0.15

_t0, _t1 = (None, None) if HIST_REGION is None else HIST_REGION
_t, vv = speed_at_dt(HIST_DT, t0=_t0, t1=_t1)
vv = np.abs(vv)
vv = vv[(vv > 5) & (vv < np.percentile(vv, 99.5))]

fig, ax_hist = plt.subplots(figsize=(11, 4.5), dpi=130)
peaks_hz = np.array([])
if len(vv) > 30:
    ax_hist.hist(vv, bins=80, density=True, histtype="stepfilled",
                 color="tab:blue", alpha=0.35, edgecolor="tab:blue")
    g = np.linspace(vv.min(), vv.max(), 500)
    k = gaussian_kde(vv, bw_method=HIST_KDE_BW)(g)
    ax_hist.plot(g, k, color="k", lw=2.0, label="KDE")
    pk, _ = find_peaks(k, prominence=k.max() * 0.05)
    peaks_hz = g[pk]
    for p in peaks_hz:
        ax_hist.axvline(p, color="tab:red", ls=":", lw=1.2)
        ax_hist.text(p, ax_hist.get_ylim()[1] * 0.92, f"{p:.0f}",
                     rotation=90, va="top", ha="right", fontsize=8, color="tab:red")
_reg = "whole file" if HIST_REGION is None else f"{HIST_REGION[0]:.0f}–{HIST_REGION[1]:.0f}s"
ax_hist.set_xlabel("|speed| (Hz)"); ax_hist.set_ylabel("density")
ax_hist.set_title(f"speed histogram ({_reg}, dt={HIST_DT*1e3:.0f} ms) — "
                  f"peaks @ " + ", ".join(f"{p:.0f}" for p in peaks_hz) + " Hz")
ax_hist.legend(fontsize=8); ax_hist.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(_output_dir / "03_speed_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {_output_dir / '03_speed_histogram.png'}")
print(f"KDE peaks (Hz): {np.round(peaks_hz, 1)}")


saved: results/files3_first40s/03_speed_histogram.png
KDE peaks (Hz): [292.6]
